# A-nanTa V7.3.1.4 — Strict BTCUSD-only training

One and only one market source: Binance COIN-M BTCUSD_PERP. No ETH, SOL, Nasdaq, NSX, or other cross-market inputs are fetched, required, cached, or consumed. The full neural architecture, 20 outputs, BF16 and eight-stage 28-epoch training schedule are retained; dynamic cross-asset Graph skips its unused neighbor branch. Full BTC M5/M15/H1/H4/D1 features remain.


In [ ]:
# Cell 1 — dependencies and reproducibility
import os, json, math, random, hashlib
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd
try:
    import pyarrow  # parquet reader
except ImportError:
    import sys, subprocess
    if 'google.colab' not in sys.modules:
        raise ImportError('pyarrow is required: pip install pyarrow')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'])
    import pyarrow
import torch
from torch import nn
from torch.nn import functional as F
SEED=2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:',torch.__version__,'device:',DEVICE)
# Optimized precision: NVIDIA L4 supports BF16 tensor cores; no GradScaler needed for BF16.
USE_BF16=(DEVICE.type=='cuda' and torch.cuda.is_bf16_supported())
from contextlib import nullcontext
def amp_context():
    return torch.autocast(device_type='cuda',dtype=torch.bfloat16) if USE_BF16 else nullcontext()
print('BF16 autocast enabled:',USE_BF16)


In [ ]:
# Config — set DATA_DIR to your mounted Google Drive Parquet folder.
CFG={
 'data_dir':'/content/manticore_v730_crypto_m5',
 'data_glob':None,'use_google_drive':True,'assets':['BTCUSD'],
 'pool_assets':['BTCUSD'],'target_asset':'BTCUSD','context_assets':[],
 'cache_dir':'/content/manticore_v730_pool_finetune_cache','output_dir':'/content/drive/MyDrive/HERMES_Colab/ai_training/manticore_v70_trend_specialist_v2/v7314_training_quality_checkpoints',
 'timestamp_semantics':'open','bar_minutes':5,'horizons':[1,3,12,48],'lookback':128,'purge_bars':192,
  'binance_market':'futures/cm','binance_symbol':'BTCUSD_PERP',
  'binance_contract_usd':100.0,'binance_auto_download':True,
 'dollar_bar_target_usd':100000.0,
 'verify_binance_sha256':True, 'checkpoint_every_steps':250,
 'validation_report_months':2, 'data_pipeline_version':'v7313_4',
 'trailing_feature_window':200,
  'binance_start':'2020-10','binance_end':None,
  'binance_archive_dir':'/content/drive/MyDrive/HERMES_Colab/ai_training/binance_vision/cm_BTCUSD_PERP_aggTrades',
  'tick_bar_cache':'/content/drive/MyDrive/HERMES_Colab/ai_training/historical_market_cache/binance_cm_btcusd_aggtrades/BTCUSD_M5.parquet',
 'train_years':5,'val_months':6,'cal_months':3,'sealed_months':3,
 'recent_half_life_days':270.,'recent_uniform_floor':0.15,
 'stress_boost':2.0,'stress_vol_quantile':0.995,'stress_cap':3.0,
 'batch_size':64,'lr':3e-4,
 'graph_history_points':8,'graph_history_stride':4,'path_rank':4,'path_knots':[1,2,3,4,6,8,12,16,24,32,40,48],
 'mtf_timeframes':{'M15':'15min','H1':'1h','H4':'4h','D1':'1D'},'mtf_window':32,'mtf_encoder_layers':2,
 'hidden':192,'transformer_layers':5,'attention_heads':6,'gru_layers':2,'graph_layers':2,'expert_width':512,'experts':8,'top_k':2,'loader_workers':2,'prefetch_factor':2,'persistent_workers':True,'graph_index_cache':True,
  'use_bf16':True,
 'neutral_move_bps':5.,'label_vol_multiplier':.25,
 'barrier_vol_multiplier':1.,'reference_cost_bps':3.,
 'min_rows':700,'min_train_sequences':100,
 'max_train_steps_per_epoch':None,  # optional bounded research runs; no silent caps
 'direction_weight':1.,'decision_weight':1.,'quantile_weight':1.,
 'rv_weight':.5,'excursion_weight':.5,'barrier_weight':.5,'regime_weight':.25,
 'route_balance_weight':.02,'pattern_weight':.35,'jump_weight':.25,'path_weight':.35,
 
 'stage_epochs':{'A_SSL':5,'B_REGIME':3,'C_GRAPH':3,'D_FORECAST':5,'E_PATH':4,'F_EXPERTS':4,'G_DECISION':2,'H_JOINT':2},
 'stage_lr_mult':{'A_SSL':1.,'B_REGIME':.65,'C_GRAPH':.5,'D_FORECAST':.5,'E_PATH':.35,'F_EXPERTS':.35,'G_DECISION':.5,'H_JOINT':.15},
 'stage_patience':{'A_SSL':3,'B_REGIME':2,'C_GRAPH':2,'D_FORECAST':3,'E_PATH':3,'F_EXPERTS':3,'G_DECISION':2,'H_JOINT':2},
 'stage_max_val_steps':None, # None means full validation
 'research_adverse_penalty':0.20, 
 'path_nll_weight':.20,'action_q_weight':.35,'pattern_detail_weight':.25,
 'use_pcgrad_joint':True,'expert_val_batches':3,
 'validate_expert_ablation':True,
 'diagnostic_gradient_assert':True,
 'run_sealed_test':False,  # explicit one-time review after freeze
}
assert 3<=CFG['train_years']<=5 and CFG['recent_half_life_days']>0
assert 0<CFG['recent_uniform_floor']<1 and CFG['stress_boost']>=1
assert max(CFG['horizons'])<CFG['purge_bars']

assert CFG['dollar_bar_target_usd']>0
assert CFG['assets']==CFG['pool_assets']==['BTCUSD'], 'BTC-only supervised asset contract'
assert CFG['target_asset']=='BTCUSD' and CFG['context_assets']==[], 'No non-BTC input assets'
assert CFG['path_knots'][-1] == max(CFG['horizons']) and sorted(set(CFG['path_knots'])) == CFG['path_knots']
# User switch: keep FP32 when BF16 is disabled even on L4.
USE_BF16=bool(USE_BF16 and CFG.get('use_bf16',True))


## 1. Market data + causal quantitative features
CSV format: `timestamp,open,high,low,close` with optional `volume,bid,ask`; file basename is asset identifier. Timezone is UTC. All rolling transforms use available history only. Missing bid/ask remain missing; they are **not fabricated**. The model uses unitless market features for cross-asset portability.
**Data source:** Binance Vision COIN-M BTCUSD_PERP aggregate executions; OHLC represents trade prices, not Bid/Ask. `volume` is aggregate USD notional (not BTC).


In [ ]:
# Cell 3 — validate data and calculate causal features
FEATURES=['ret1','ret3','ret12','vol12','vol48','vol_ratio','efficiency','range_norm','position','downside','skew24','kurt24','entropy24','spread_rel','spread_present','volume_log','volume_present','trend_strength_96','volatility_har_12','volatility_har_48','volatility_har_288','hurst_proxy','wavelet_band','jump_proxy','mean_reversion_z','autocorr24','taker_dollar_imbalance','taker_buy_ratio','notional_log','trade_count_log','imbalance_ewm','flow_shock','dollar_event_count_log','dollar_event_imbalance','dollar_event_activity']
# Asset identity is intentionally NOT used by the model: new assets use the same causal statistics.
def load_asset_file(path):
    """Read verified local-mounted Drive files; no external downloads and no CSV conversion."""
    path=Path(path)
    if path.suffix.lower()=='.parquet':
        d=pd.read_parquet(path)
    elif path.suffix.lower()=='.csv':
        d=pd.read_csv(path)
    else:
        raise ValueError(f'Unsupported market file: {path}')
    d.columns=[str(c).lower().strip() for c in d.columns]
    if 'timestamp' not in d.columns:
        for key in ('datetime','date_time','time','date'):
            if key in d.columns:
                d=d.rename(columns={key:'timestamp'});break
    if 'timestamp' not in d.columns and isinstance(d.index,pd.DatetimeIndex):
        d=d.reset_index().rename(columns={d.index.name or 'index':'timestamp'})
    if 'timestamp' in d.columns and pd.api.types.is_numeric_dtype(d['timestamp']):
        raise ValueError(f'{path}: numeric epoch timestamps need explicit unit (s/ms/ns); aborting rather than guessing')
    missing={'timestamp','open','high','low','close'}-set(d.columns)
    if missing:raise ValueError(f'{path}: missing {sorted(missing)}')
    d['timestamp']=pd.to_datetime(d['timestamp'],utc=True,errors='raise')
    if d['timestamp'].isna().any() or d['timestamp'].duplicated().any():
        raise ValueError(f'{path}: null/duplicate timestamps; fix raw source, do not silently deduplicate')
    d=d.sort_values('timestamp').set_index('timestamp')
    for k in ('open','high','low','close','bid','ask','volume','notional_usd','signed_notional_usd','buy_notional_usd','trade_count'):
        if k in d:d[k]=pd.to_numeric(d[k],errors='coerce')
    if d[['open','high','low','close']].isna().any().any():raise ValueError(f'{path}: missing prices')
    if len(d)<CFG['min_rows']:raise ValueError(f'{path}: insufficient bars {len(d)}')
    if ((d[['open','high','low','close']]<=0).any().any() or (d.high<d[['open','close','low']].max(axis=1)).any()
            or (d.low>d[['open','close','high']].min(axis=1)).any()):
        raise ValueError(f'{path}: invalid OHLC')
    if {'bid','ask'}<=set(d.columns) and ((d.ask<d.bid)&d.ask.notna()&d.bid.notna()).any():
        raise ValueError(f'{path}: crossed quotes')
    # Non-uniform timestamps are allowed, but the report must disclose cadence/gaps per asset.
    return d

def fast_permutation_entropy(log_returns,window=24):
    """Exact causal order-3 permutation entropy over each 24-return window;
    rolling sample contains window-2 consecutive triplets. O(N), six permutations.
    """
    a=np.asarray(log_returns,dtype=np.float64)
    x=a[:-2];y=a[1:-1];z=a[2:]
    # stable ties: earlier temporal element wins
    rank_x=((y<x).astype('int8')+(z<x).astype('int8'))
    rank_y=((x<=y).astype('int8')+(z<y).astype('int8'))
    rank_z=3-rank_x-rank_y
    # Each permutation identified by base-3 encoded ranks, six distinct values.
    patterns=np.full(len(a),-1,dtype=np.int16)
    good=np.isfinite(x)&np.isfinite(y)&np.isfinite(z)
    patterns[2:][good]=(rank_x[good]*9+rank_y[good]*3+rank_z[good]).astype(np.int16)
    out=np.full(len(a),np.nan,dtype=np.float64)
    count=np.zeros(len(a),dtype=np.int16)
    ent=np.zeros(len(a),dtype=np.float64)
    # 22 consecutive patterns in a 24-return window, causal at index t.
    for key in (5,7,11,15,19,21):
        hit=pd.Series((patterns==key).astype('int8'))
        count_arr=hit.rolling(window-2,min_periods=window-2).sum().to_numpy()
        count+=np.nan_to_num(count_arr,nan=0).astype(np.int16)
        prob=np.nan_to_num(count_arr,nan=0)/(window-2)
        ent-=np.where(prob>0,prob*np.log(np.maximum(prob,1e-30)),0)
    # exclude any missing triplets in rolling window
    valid=pd.Series((patterns>=0).astype('int8')).rolling(window-2,min_periods=window-2).sum().to_numpy()==window-2
    out[valid]=ent[valid]/np.log(6)
    return out

def _causal_features_one_contiguous_segment(d):
    p=d.close;lr=np.log(p).diff();v=lr.rolling(48,min_periods=24).std()
    f=pd.DataFrame(index=d.index)
    f['ret1']=lr;f['ret3']=np.log(p/p.shift(3));f['ret12']=np.log(p/p.shift(12))
    f['vol12']=lr.rolling(12,min_periods=12).std();f['vol48']=v
    f['vol_ratio']=f.vol12/(v+1e-8)
    f['efficiency']=p.diff(24).abs()/(p.diff().abs().rolling(24).sum()+1e-8)
    f['range_norm']=(d.high-d.low)/(p*v.replace(0,np.nan)+1e-8)
    f['position']=(p-p.rolling(24).min())/(p.rolling(24).max()-p.rolling(24).min()+1e-8)
    f['downside']=lr.clip(upper=0).pow(2).rolling(24).mean().pow(.5)
    f['skew24']=lr.rolling(24).skew();f['kurt24']=lr.rolling(24).kurt()
    f['entropy24']=fast_permutation_entropy(lr.to_numpy(),24)
    if {'bid','ask'}<=set(d.columns):
        raw=((d.ask-d.bid)/p).where((d.ask>=d.bid)&(d.bid>0))
        f['spread_rel']=raw;f['spread_present']=raw.notna().astype(float)
    else:f['spread_rel']=np.nan;f['spread_present']=0.
    if 'volume' in d:
        x=np.log1p(d.volume.clip(lower=0));f['volume_log']=x-x.rolling(48,min_periods=24).mean()
        f['volume_present']=d.volume.notna().astype(float)
    else:f['volume_log']=np.nan;f['volume_present']=0.
    # Causal multiscale and distributional Quant state features (no future data).
    f['trend_strength_96']=np.log(p/p.shift(96))/(lr.rolling(96,min_periods=48).std()*np.sqrt(96)+1e-9)
    f['volatility_har_12']=lr.pow(2).rolling(12,min_periods=12).mean()
    f['volatility_har_48']=lr.pow(2).rolling(48,min_periods=48).mean()
    f['volatility_har_288']=lr.pow(2).rolling(288,min_periods=144).mean()
    f['hurst_proxy']=np.log((lr.rolling(48,min_periods=24).sum().abs()+1e-9)/
                   (lr.abs().rolling(48,min_periods=24).sum()+1e-9))
    f['wavelet_band']=lr.rolling(8,min_periods=8).mean()-lr.rolling(48,min_periods=24).mean()
    f['jump_proxy']=(lr.abs()/(lr.abs().rolling(48,min_periods=24).median()+1e-9)).clip(upper=50)
    f['mean_reversion_z']=(np.log(p)-np.log(p).rolling(48,min_periods=24).mean())/(v*np.sqrt(48)+1e-8)
    f['autocorr24']=lr.rolling(24,min_periods=24).corr(lr.shift(1))
    # Aggregated executions from Binance COIN-M Futures. 100 USD/contract for BTCUSD_PERP;
    # is_buyer_maker=True means aggressive SELL. These are observed trade flows, not liquidations.
    notional=d['notional_usd'] if 'notional_usd' in d else pd.Series(np.nan,index=d.index)
    signed=d['signed_notional_usd'] if 'signed_notional_usd' in d else pd.Series(np.nan,index=d.index)
    buys=d['buy_notional_usd'] if 'buy_notional_usd' in d else pd.Series(np.nan,index=d.index)
    trades=d['trade_count'] if 'trade_count' in d else pd.Series(np.nan,index=d.index)
    imbalance=(signed/(notional+1e-9)).where(notional>0)
    f['taker_dollar_imbalance']=imbalance
    f['taker_buy_ratio']=(buys/(notional+1e-9)).where(notional>0)
    f['notional_log']=np.log1p(notional.clip(lower=0))-np.log1p(notional.clip(lower=0)).rolling(48,min_periods=24).mean()
    f['trade_count_log']=np.log1p(trades.clip(lower=0))-np.log1p(trades.clip(lower=0)).rolling(48,min_periods=24).mean()
    f['imbalance_ewm']=imbalance.ewm(span=24,adjust=False,min_periods=12).mean()
    f['flow_shock']=(signed.abs()/(signed.abs().rolling(48,min_periods=24).median()+1e-9)).clip(0,50)
    # Dollar-imbalance events: threshold-crossing of executed USD notional, aligned to CLOSED M5 bar.
    # Each event was completed no later than this bar, and no future quote is used.
    events=d['dollar_event_count'] if 'dollar_event_count' in d else pd.Series(np.nan,index=d.index)
    signed_events=d['dollar_event_imbalance'] if 'dollar_event_imbalance' in d else pd.Series(np.nan,index=d.index)
    f['dollar_event_count_log']=np.log1p(events.clip(lower=0))
    f['dollar_event_imbalance']=signed_events
    f['dollar_event_activity']=events.clip(lower=0)/(events.clip(lower=0).rolling(48,min_periods=24).mean()+1e-9)
    return f.replace([np.inf,-np.inf],np.nan)

def causal_features(d):
    """Causal features must not bridge missing source candles or market closures."""
    if d.empty:return _causal_features_one_contiguous_segment(d)
    times=d.index.to_numpy(dtype='datetime64[ns]')
    if len(times)<2:return _causal_features_one_contiguous_segment(d)
    steps=np.diff(times).astype('timedelta64[ns]').astype(np.int64)
    if np.any(steps<=0):raise ValueError('Non-monotonic feature source timestamps')
    expected=int(np.min(steps)) # 5m on source bars; e.g. 15m/1h on resampled inputs.
    boundaries=np.r_[0,np.where(steps!=expected)[0]+1,len(d)]
    if len(boundaries)==2:return _causal_features_one_contiguous_segment(d)
    parts=[_causal_features_one_contiguous_segment(d.iloc[a:b]) for a,b in zip(boundaries[:-1],boundaries[1:])]
    return pd.concat(parts).reindex(d.index)


In [ ]:
# Cell 4 — future-only supervised targets. Inputs never use these columns.
# Regime targets are CAUSAL heuristic teachers (trend persistence, volatility, transition).
# Decision teacher uses future FIRST-TOUCH barrier utility, NOT direction classification.
def build_labels(d,horizons):
    lr=np.log(d.close);r=lr.diff();y=pd.DataFrame(index=d.index)
    local_vol=r.rolling(48,min_periods=24).std().clip(lower=1e-8)
    eff=(d.close.diff(24).abs()/(d.close.diff().abs().rolling(24).sum()+1e-8)).fillna(0)
    vratio=(r.rolling(12,min_periods=12).std()/(local_vol+1e-8)).fillna(1)
    y['regime_trend']=eff.clip(0,1) # an explicit causal weak teacher, not ground truth
    y['regime_high_vol']=(vratio>1.5).astype(float)
    y['regime_transition']=((eff-eff.shift(12)).abs()>.2).astype(float)
    squared=r.pow(2)
    for h in horizons:
        future=lr.shift(-h)-lr
        y[f'ret_{h}']=future
        threshold=np.maximum(CFG['neutral_move_bps']*1e-4,CFG['label_vol_multiplier']*local_vol*np.sqrt(h))
        y[f'dir_{h}']=np.where(future>threshold,2,np.where(future< -threshold,0,1)).astype(float)
        y.loc[future.isna(),f'dir_{h}']=np.nan
        # Require COMPLETE future window. max/min must not be computed on an incomplete tail.
        base_close=d.close.to_numpy(dtype=np.float64)
        hi=d.high.to_numpy(dtype=np.float64);lo=d.low.to_numpy(dtype=np.float64)
        max_exc=np.full(len(d),-np.inf);min_exc=np.full(len(d),np.inf)
        valid=np.ones(len(d),dtype=bool)
        upper=(CFG['barrier_vol_multiplier']*local_vol*np.sqrt(h)).clip(lower=CFG['neutral_move_bps']*1e-4).to_numpy()
        up=base_close*np.exp(upper);dn=base_close*np.exp(-upper)
        barrier=np.ones(len(d),dtype=np.int8) # 0 SHORT, 1 WAIT, 2 LONG
        undecided=np.isfinite(upper)&(upper>=CFG['reference_cost_bps']*1e-4)
        ambiguous=np.zeros(len(d),dtype=bool)
        for j in range(1,h+1):
            fhi=np.full(len(d),np.nan);flo=np.full(len(d),np.nan)
            fhi[:-j]=hi[j:];flo[:-j]=lo[j:]
            valid &= np.isfinite(fhi)&np.isfinite(flo)
            with np.errstate(divide='ignore',invalid='ignore'):
                max_exc=np.maximum(max_exc,np.log(fhi/base_close))
                min_exc=np.minimum(min_exc,np.log(flo/base_close))
            both=(fhi>=up)&(flo<=dn)&undecided
            only_up=(fhi>=up)&~both&undecided
            only_dn=(flo<=dn)&~both&undecided
            ambiguous |= both
            barrier[only_up]=2;barrier[only_dn]=0
            undecided[only_up|only_dn|both]=False
        y[f'mfe_{h}']=np.where(valid,max_exc,np.nan)
        y[f'mae_{h}']=np.where(valid,min_exc,np.nan)
        # Future realized variance: squared future close returns, no future values in inputs
        y[f'rv_{h}']=sum(squared.shift(-j) for j in range(1,h+1))
        y[f'barrier_{h}']=np.where(ambiguous,np.nan,barrier.astype(float))
        # Same-bar high/low cannot establish execution order; exclude this supervised sample.
        # Event outcomes: first-touch breakout, reversal and jump within horizon.
        past_high=d.high.rolling(48,min_periods=24).max().shift(1).to_numpy(dtype=np.float64)
        past_low=d.low.rolling(48,min_periods=24).min().shift(1).to_numpy(dtype=np.float64)
        # Future maximum/minimum log excursions were computed only for targets.
        y[f'pattern_{h}']=np.where(np.isfinite(past_high)&np.isfinite(past_low),
            ((base_close*np.exp(max_exc)>past_high)|(base_close*np.exp(min_exc)<past_low)).astype(float),np.nan)
        y[f'jump_{h}']=np.where(valid,((max_exc-min_exc)>3.5*np.maximum(local_vol.to_numpy()*np.sqrt(h),1e-8)).astype(float),np.nan)
        # Coarse path-shape target: final displacement relative to maximum signed excursion.
        y[f'path_{h}']=np.where(valid,np.clip(future.to_numpy()/(np.maximum(max_exc-min_exc,1e-8)),-2,2),np.nan)
        y.iloc[-h:,y.columns.get_loc(f'barrier_{h}')]=np.nan
        # A row count is NOT a time horizon when source M5 is missing.
        # Monotonic, unique timestamps plus exact elapsed time prove every 5-minute bar exists.
        delta=np.timedelta64(h*CFG['bar_minutes'],'m')
        time_utc=d.index.to_numpy(dtype='datetime64[ns]')
        connected=np.zeros(len(d),dtype=bool)
        connected[:-h]=(time_utc[h:]-time_utc[:-h])==delta
        for suffix in ('ret','dir','mfe','mae','rv','barrier','pattern','jump','path'):
            y.loc[~connected,f'{suffix}_{h}']=np.nan

    # LONG/SHORT/WAIT decision teacher uses first-touch event at longest horizon.
    # This is a noisy research target, NOT an executable trading policy or broker simulation.
    y['decision']=y[f'barrier_{horizons[-1]}']
    # Event pattern targets (future-only): false breakout, trend reversal, trend continuation.
    past_mom=np.log(d.close/d.close.shift(24)).to_numpy(dtype=np.float64)
    for h in horizons:
        complete=y[f'ret_{h}'].notna().to_numpy()
        ph=d.high.rolling(48,min_periods=24).max().shift(1).to_numpy(dtype=np.float64)
        pl=d.low.rolling(48,min_periods=24).min().shift(1).to_numpy(dtype=np.float64)
        terminal=d.close.shift(-h).to_numpy(dtype=np.float64)
        mfe=y[f'mfe_{h}'].to_numpy(dtype=np.float64)
        mae=y[f'mae_{h}'].to_numpy(dtype=np.float64)
        valid=complete&np.isfinite(ph)&np.isfinite(pl)&np.isfinite(past_mom)
        touched=((d.close.to_numpy()*np.exp(mfe)>ph)|(d.close.to_numpy()*np.exp(mae)<pl))
        inside=(terminal<=ph)&(terminal>=pl)
        y[f'false_breakout_{h}']=np.where(valid,(touched&inside).astype(float),np.nan)
        # Volatility-normalized patterns. No subjective chart labels.
        future=y[f'ret_{h}'].to_numpy(dtype=np.float64)
        move_cut=(CFG['label_vol_multiplier']*local_vol*np.sqrt(h)).to_numpy(dtype=np.float64)
        past_signal=np.sign(past_mom)
        real_move=np.abs(future)>np.maximum(move_cut,CFG['neutral_move_bps']*1e-4)
        y[f'reversal_{h}']=np.where(valid,(real_move&(future*past_signal<0)).astype(float),np.nan)
        y[f'continuation_{h}']=np.where(valid,(real_move&(future*past_signal>0)).astype(float),np.nan)
        time_utc=d.index.to_numpy(dtype='datetime64[ns]')
        connected=np.zeros(len(d),dtype=bool)
        connected[:-h]=(time_utc[h:]-time_utc[:-h])==np.timedelta64(h*CFG['bar_minutes'],'m')
        for suffix in ('false_breakout','reversal','continuation'):
            y.loc[~connected,f'{suffix}_{h}']=np.nan

    # Stochastic path is trained on log-return INCREMENTS at fixed future knots.
    previous=0
    for k in CFG['path_knots']:
        y[f'path_increment_{k}']=np.log(d.close.shift(-k)/d.close.shift(-previous))
        time_utc=d.index.to_numpy(dtype='datetime64[ns]')
        connected=np.zeros(len(d),dtype=bool)
        connected[:-k]=(time_utc[k:]-time_utc[:-k])==np.timedelta64(k*CFG['bar_minutes'],'m')
        y.loc[~connected,f'path_increment_{k}']=np.nan
        previous=k
    return y


## 2. Temporal validation discipline

Four global calendar partitions: **train / validation / calibration / sealed OOS**. Purge at every fold boundary and require all target horizons to fall inside the same fold. All feature scaler statistics are fitted on TRAIN only. The calibration partition is used only after model selection; sealed test is consumed once. For a second experiment, create a NEW sealed interval, never repeatedly optimize the old one.

In [ ]:
# Binance Vision COIN-M BTCUSD_PERP aggregate executions. Download once, cache in Drive,
# aggregate into UTC M5 OHLC and signed flow.  aggTrades are grouped executions, not raw ticks.
import glob, gc, csv, io, zipfile, re, urllib.request, urllib.error, time, hashlib
if CFG['use_google_drive'] and Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')

AGG_COLS=['agg_trade_id','price','quantity','first_trade_id','last_trade_id','timestamp','is_buyer_maker']
ARCHIVE=Path(CFG['binance_archive_dir'])
ARCHIVE.mkdir(parents=True,exist_ok=True)

def _month_range(start,stop):
    a=pd.Period(start,freq='M');b=pd.Period(stop,freq='M')
    return [str(p) for p in pd.period_range(a,b,freq='M')]

def _binance_checksum_text(url):
    req=urllib.request.Request(url+'.CHECKSUM',headers={'User-Agent':'MANTICORE-research/1.0'})
    with urllib.request.urlopen(req,timeout=90) as response:
        line=response.read(512).decode('ascii').strip()
    digest=line.split()[0].lower() if line else ''
    if not re.fullmatch(r'[a-f0-9]{64}',digest):
        raise ValueError(f'Missing valid Binance SHA256 CHECKSUM: {url}.CHECKSUM')
    return digest

def _sha256_file(p,chunk_size=4*1024*1024):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for part in iter(lambda:f.read(chunk_size),b''):
            h.update(part)
    return h.hexdigest()

def verify_binance_zip(path,url):
    path=Path(path)
    if not zipfile.is_zipfile(path):raise ValueError(f'Invalid ZIP: {path}')
    digest=_binance_checksum_text(url)
    marker=path.with_suffix(path.suffix+'.sha256_verified.json')
    stat=path.stat()
    try:
        cached=json.loads(marker.read_text())
        good=cached['expected_sha256']==digest and cached['size']==stat.st_size and cached['mtime_ns']==stat.st_mtime_ns
    except (OSError,ValueError,KeyError):good=False
    if not good:
        actual=_sha256_file(path)
        if actual!=digest:raise ValueError(f'Binance SHA256 mismatch for {path}: {actual} != {digest}')
        with zipfile.ZipFile(path) as z:
            broken=z.testzip()
            if broken:raise ValueError(f'Binance ZIP CRC mismatch {path}: {broken}')
        marker.write_text(json.dumps({'expected_sha256':digest,'size':stat.st_size,'mtime_ns':stat.st_mtime_ns}))
    return digest

def fetch_binance_month(month,sym=None,market=None,archive=None):
    sym=sym or CFG['binance_symbol'];market=market or CFG['binance_market'];archive=Path(archive or ARCHIVE);archive.mkdir(parents=True,exist_ok=True)
    name=f'{sym}-aggTrades-{month}.zip'
    path=archive/name
    url=f'https://data.binance.vision/data/{market}/monthly/aggTrades/{sym}/{name}'
    if path.exists() and zipfile.is_zipfile(path):
        if CFG['verify_binance_sha256']:verify_binance_zip(path,url)
        return path
    if not CFG['binance_auto_download']:
        raise FileNotFoundError(f'Missing {path}; turn on binance_auto_download')
    tmp=path.with_suffix('.download')
    for attempt in range(4):
        try:
            req=urllib.request.Request(url,headers={'User-Agent':'MANTICORE-research/1.0'})
            with urllib.request.urlopen(req,timeout=120) as resp, open(tmp,'wb') as f:
                for block in iter(lambda:resp.read(2*1024*1024),b''):
                    f.write(block)
            if not zipfile.is_zipfile(tmp):raise ValueError(f'Bad ZIP: {url}')
            with zipfile.ZipFile(tmp) as z:
                bad=z.testzip()
                if bad:raise ValueError(f'ZIP corruption: {bad}')
            # Validate before moving to final cache location.
            if CFG['verify_binance_sha256']:
                expected=_binance_checksum_text(url)
                if _sha256_file(tmp)!=expected:raise ValueError(f'SHA256 check failed for {url}')
            tmp.replace(path)
            if CFG['verify_binance_sha256']:verify_binance_zip(path,url)
            return path
        except urllib.error.HTTPError as e:
            if e.code==404:raise FileNotFoundError(f'Binance Vision monthly file unavailable: {url}') from e
            if attempt==3:raise
        except Exception:
            if attempt==3:raise
        time.sleep(2**attempt)
    raise RuntimeError(f'Cannot download: {url}')

def iter_aggtrades(path,chunk_size=350_000):
    with zipfile.ZipFile(path) as z:
        members=[m for m in z.namelist() if m.lower().endswith('.csv') and not m.endswith('/')]
        if len(members)!=1:raise ValueError(f'Expected 1 aggTrades CSV in {path}: {members}')
        with z.open(members[0]) as f:
            # Binance archive switched some derivatives CSVs from no-header to header;
            # detect exact 7-column header, rather than accidentally parse it as a trade.
            first=f.readline().decode('utf-8-sig').strip().lower()
            f.seek(0)
            has_header=first.startswith('agg_trade_id,') or first.startswith('aggtradeid,') or first.startswith('aggtrade_id,')
            if not has_header and ('timestamp' in first or 'price' in first) and not first[0].isdigit():has_header=True
            for chunk in pd.read_csv(f,header=None,names=AGG_COLS,skiprows=1 if has_header else 0,
                                     chunksize=chunk_size,on_bad_lines='error'):
                yield chunk

def parse_agg_chunk(c,path,market='futures/cm'):
    for key in ('price','quantity','timestamp','agg_trade_id'):
        c[key]=pd.to_numeric(c[key],errors='raise')
    if c[['price','quantity','timestamp','agg_trade_id']].isna().any().any():raise ValueError(f'{path}: null trade fields')
    if (c.price<=0).any() or (c.quantity<=0).any():raise ValueError(f'{path}: bad price or quantity')
    maker=c.is_buyer_maker.astype(str).str.lower()
    if not maker.isin(['true','false','1','0']).all():raise ValueError(f'{path}: bad maker/taker flags')
    sell=maker.isin(['true','1']).to_numpy()
    # For COIN-M inverse BTCUSD_PERP contract quantity denotes contracts, NOT BTC.
    # Contract-size must be checked against Binance contract specifications before live use.
    usd=(c.quantity.to_numpy('float64')*float(CFG['binance_contract_usd']) if market=='futures/cm'
         else c.quantity.to_numpy('float64')*c.price.to_numpy('float64'))
    signed=np.where(sell,-usd,usd)
    ts=c.timestamp.to_numpy('int64')
    if (ts<1_500_000_000_000).any() or (ts>10_000_000_000_000_000).any():
        raise ValueError(f'{path}: unexpected timestamp epoch units')
    # Older Binance archives use milliseconds; newer Spot archives can use microseconds.
    unit='us' if np.median(ts)>10_000_000_000_000 else 'ms'
    return pd.DataFrame({'timestamp':pd.to_datetime(ts,unit=unit,utc=True),
             'price':c.price.to_numpy('float64'),'notional_usd':usd,
             'signed_notional_usd':signed,'buy_notional_usd':np.where(sell,0.,usd),
             'trade_count':np.ones(len(c),dtype=np.int64),'agg_trade_id':c.agg_trade_id.to_numpy('int64')})

def convert_binance_aggtrades_to_m5(files,outfile,market='futures/cm',asset='BTCUSD'):
    bars={}; total=0;newest=None;last_id=None;last_id_timestamp=None
    id_disorders=[];id_disorder_count=0
    carry_usd=0.;carry_signed=0.;event_counts={};event_signs={};event_last_price={};event_last_close={}
    threshold=float(CFG['dollar_bar_target_usd'])
    for path in files:
        monthly=0
        for chunk in iter_aggtrades(path):
            q=parse_agg_chunk(chunk,path,market=market)
            if len(q)==0:continue
            if not q.timestamp.is_monotonic_increasing or (newest is not None and q.timestamp.min()<newest):
                raise ValueError(f'{path}: timestamp disorder; cannot construct event sequence reliably')
            # Aggregate IDs are audited, not used to order executions. Some verified
            # historical USD-M archives have nonmonotonic IDs; never silently drop,
            # deduplicate, or sort trades by ID. Timestamp reversals still fail closed.
            ids=q.agg_trade_id.to_numpy(dtype=np.int64)
            timestamps=q.timestamp.array.asi8
            bad_internal=np.flatnonzero(ids[1:]<=ids[:-1])+1
            bad_boundary=last_id is not None and int(ids[0])<=last_id
            count=int(len(bad_internal))+int(bad_boundary)
            if count:
                id_disorder_count+=count
                samples=([0] if bad_boundary else [])+bad_internal[:max(0,8-len([0] if bad_boundary else []))].tolist()
                for pos in samples:
                    if len(id_disorders)>=32:break
                    previous_id=last_id if pos==0 else int(ids[pos-1])
                    previous_time=last_id_timestamp if pos==0 else str(q.timestamp.iloc[pos-1])
                    id_disorders.append({'file':Path(path).name,'row_in_chunk':int(pos),
                        'previous_id':int(previous_id),'current_id':int(ids[pos]),
                        'previous_timestamp':previous_time,'timestamp':str(q.timestamp.iloc[pos])})
                print('DATA QUALITY WARNING:',asset,Path(path).name,
                      'aggregate_trade_id non-increasing transitions:',count,
                      '(source order retained; verify audit report)')
            newest=q.timestamp.max();last_id=int(ids[-1]);last_id_timestamp=str(q.timestamp.iloc[-1])
            monthly+=len(q);total+=len(q)
            # True notional-threshold event segmentation across chunks/months.
            # Each aggregated execution carries its maker/taker sign, but is not a raw individual trade.
            # A large aggTrade may contain several threshold units: split its dollar flow proportionally.
            q['bar']=q.timestamp.dt.floor('5min')
            # Vectorized threshold crossings: loops over EVENT bars, not every execution.
            usd_arr=q.notional_usd.to_numpy('float64')
            flow_arr=q.signed_notional_usd.to_numpy('float64')
            cs=np.cumsum(usd_arr,dtype='float64')
            cumulative=carry_usd+cs
            completed=int(np.floor((cumulative[-1]+1e-7)/threshold))
            if completed:
                levels=threshold*np.arange(1,completed+1,dtype='float64')
                ix=np.searchsorted(cumulative,levels,side='left')
                prev_dollar=np.r_[0.,cs[:-1]][ix]+carry_usd
                fraction=(levels-prev_dollar)/usd_arr[ix]
                signed_cum=carry_signed+np.r_[0.,np.cumsum(flow_arr,dtype='float64')[:-1]][ix]+fraction*flow_arr[ix]
                event_signed=np.diff(np.r_[0.,signed_cum])/threshold
                timestamp_keys=q['bar'].array.asi8[ix]
                event_prices=q.price.to_numpy('float64')[ix]
                for key,sgn,pr in zip(timestamp_keys,event_signed,event_prices):
                    key=int(key)
                    event_counts[key]=event_counts.get(key,0)+1
                    event_signs[key]=event_signs.get(key,0.)+float(sgn)
                    event_last_price[key]=float(pr)
                carry_usd=float(cumulative[-1]-completed*threshold)
                carry_signed=float(carry_signed+flow_arr.sum()-signed_cum[-1])
            else:
                carry_usd+=float(cs[-1]);carry_signed+=float(flow_arr.sum())
            g=q.groupby('bar',sort=True).agg(open=('price','first'),high=('price','max'),low=('price','min'),
                close=('price','last'),notional_usd=('notional_usd','sum'),
                signed_notional_usd=('signed_notional_usd','sum'),buy_notional_usd=('buy_notional_usd','sum'),
                trade_count=('trade_count','sum'))
            for t,row in g.iterrows():
                k=t.value;v=(float(row.open),float(row.high),float(row.low),float(row.close),
                           float(row.notional_usd),float(row.signed_notional_usd),
                           float(row.buy_notional_usd),int(row.trade_count))
                prev=bars.get(k)
                bars[k]=((prev[0],max(prev[1],v[1]),min(prev[2],v[2]),v[3],
                          prev[4]+v[4],prev[5]+v[5],prev[6]+v[6],prev[7]+v[7]) if prev else v)
        print('Binance',asset,path.name,'aggTrades:',monthly)
    if not bars:raise ValueError('No actual aggregate trades')
    records=({'timestamp':pd.Timestamp(k,unit='ns',tz='UTC'),
              'open':v[0],'high':v[1],'low':v[2],'close':v[3],
              'volume':v[4], 'notional_usd':v[4], 'signed_notional_usd':v[5],
              'buy_notional_usd':v[6], 'trade_count':v[7],
               'dollar_event_count':event_counts.get(k,0),
               'dollar_event_imbalance':event_signs.get(k,0.)/max(event_counts.get(k,0),1),
               'dollar_event_last_price':event_last_price.get(k,np.nan)} for k,v in sorted(bars.items()))
    out=Path(outfile);out.parent.mkdir(parents=True,exist_ok=True)
    pd.DataFrame(records).to_parquet(out,index=False)
    audit={'asset':asset,'market':market,'source_files':len(files),
           'agg_trades':total,'m5_bars':len(bars),
           'nonincreasing_aggregate_id_transitions':id_disorder_count,
           'aggregate_id_disorder_samples':id_disorders,
           'id_policy':'timestamp-ordered source rows; log ID anomalies; no sorting/dropping'}
    audit_path=out.with_suffix('.aggtrade_id_audit.json')
    audit_path.write_text(json.dumps(audit,indent=2))
    print('M5 bars:',len(bars),'aggTrades:',total,'saved:',out,
          'ID anomalies:',id_disorder_count,'audit:',audit_path)
    return out

# Download Binance Vision separately per market; use each market's own quantity semantics.
now=pd.Timestamp.now(tz='UTC')
last_complete=str(now.tz_localize(None).to_period('M')-1) if CFG['binance_end'] is None else CFG['binance_end']
months=_month_range(CFG['binance_start'],last_complete)

def file_fingerprint(files,kind='binance',market=None,symbol=None):
    # All source bytes are cryptographically pinned, not merely filename/mtime.
    fingerprints=[]
    for p in files:
        p=Path(p)
        if kind=='binance' and CFG['verify_binance_sha256']:
            marker=p.with_suffix(p.suffix+'.sha256_verified.json')
            verified=json.loads(marker.read_text())
            if (verified['size']!=p.stat().st_size or verified['mtime_ns']!=p.stat().st_mtime_ns):
                raise RuntimeError(f'Stale source verification marker: {p}')
            digest=verified['expected_sha256']
        else:
            digest=_sha256_file(p)
        fingerprints.append((p.name,digest))
    contract={'files':fingerprints,'kind':kind,'market':market,'symbol':symbol,
      'dollar_bar_target_usd':CFG['dollar_bar_target_usd'],
      'binance_contract_usd':CFG['binance_contract_usd'],
      'pipeline_version':CFG['data_pipeline_version'],'timestamp_semantics':CFG['timestamp_semantics']}
    return hashlib.sha256(json.dumps(contract,sort_keys=True).encode()).hexdigest()

def build_cached_binance_asset(asset,market,symbol,archive,output):
    files=[];first_month=None;missing_leading=[]
    for m in months:
        try:
            p=fetch_binance_month(m,sym=symbol,market=market,archive=archive)
        except FileNotFoundError:
            if first_month is None and asset!=CFG['target_asset']:
                missing_leading.append(m) # legitimate pre-listing months, not data gaps
                continue
            raise RuntimeError(f'Missing Binance month INSIDE active {asset} history: {m}')
        files.append(p)
        if first_month is None:first_month=m
    if not files:raise ValueError(f'No Binance monthly aggTrades archives for {asset}')
    if missing_leading:print(f'{asset}: skipped {len(missing_leading)} pre-listing months; first available {first_month}')
    fingerprint=file_fingerprint(files,kind="binance",market=market,symbol=symbol)
    path=Path(output);manifest=path.with_suffix('.source_manifest.json')
    try:reuse=path.exists() and json.loads(manifest.read_text())['hash']==fingerprint
    except (OSError,ValueError,KeyError):reuse=False
    if not reuse:
        convert_binance_aggtrades_to_m5(files,path,market=market,asset=asset)
        manifest.write_text(json.dumps({'hash':fingerprint,'market':market,'symbol':symbol,
            'agg_trades':'grouped executions','liquidation_labels':False,
            'notional':'COIN-M qty*contract_usd; USD-M qty*price',
            'dollar_event_target_usd':CFG['dollar_bar_target_usd']},indent=2))
    else:print('Reusing cached',asset,path)
    return path

# Strict BTC-only source: never request, download, parse, or join other assets.
btc_path=build_cached_binance_asset('BTCUSD',CFG['binance_market'],CFG['binance_symbol'],
    CFG['binance_archive_dir'],CFG['tick_bar_cache'])
paths=[btc_path]
assert len(paths)==1 and paths[0].exists(), 'BTC-only source required'
coverage={};time_bounds=[]
for path in paths:
    d=load_asset_file(path);asset=path.stem.replace('_M5','').upper()
    gaps=d.index.to_series().diff().dt.total_seconds()
    coverage[asset]={'bars':len(d),'start':str(d.index.min()),'end':str(d.index.max()),
         'non_5m_intervals':int((gaps.dropna()!=300).sum()),
         'largest_gap_seconds':float(gaps.max()) if len(gaps.dropna()) else 0.,'source':str(path),
         'role':'pooled_train' if asset in CFG['pool_assets'] else 'graph_context_only'}
    if asset==CFG['target_asset']:time_bounds.append((d.index.min(),d.index.max()))
    del d,gaps;gc.collect()
start,end=time_bounds[0]
sealed_start=end-pd.DateOffset(months=int(CFG['sealed_months']))
cal_start=sealed_start-pd.DateOffset(months=int(CFG['cal_months']))
train_end=cal_start-pd.DateOffset(months=int(CFG['val_months']))
train_start=train_end-pd.DateOffset(years=int(CFG['train_years']))
cuts=[train_end,cal_start,sealed_start]
for asset in CFG['pool_assets']:
    begin=pd.Timestamp(coverage[asset]['start'])
    if begin>train_start:print('WARNING:',asset,'has shorter available training history:',begin)
if train_start<start:raise RuntimeError('Insufficient BTCUSD history for 5y train + 12m holdout')
CFG['train_start_utc']=str(train_start);CFG['train_end_utc']=str(train_end)
print('STRICT UTC folds:',train_start,cuts,end,'\nCoverage:',json.dumps(coverage,indent=2))


In [ ]:
# Cell 6 — Per-asset disk-backed features/targets and lazy sample indices
from torch.utils.data import Dataset,DataLoader
import gc
# Cache namespace includes input file content fingerprint and preprocessing contract.
# Avoid stale NumPy arrays when source files or label/feature pipeline changes.
def _source_fingerprint(p,chunk=2**20):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b:break
            h.update(b)
    return h.hexdigest()
_cache_inputs=[(str(p),_source_fingerprint(p)) for p in paths]
_cache_contract={'sources':_cache_inputs,'features':FEATURES,
    'code_version':'v7313_contiguous_masked_labels_isolated_validation',
    'cfg':{k:CFG[k] for k in ('lookback','purge_bars','horizons','train_years','val_months',
       'cal_months','sealed_months','bar_minutes','timestamp_semantics','mtf_timeframes',
       'mtf_window','graph_history_points','graph_history_stride',
       'stress_vol_quantile','barrier_vol_multiplier','neutral_move_bps',
       'label_vol_multiplier','reference_cost_bps','path_knots','validation_report_months','trailing_feature_window')}}
CACHE_FINGERPRINT=hashlib.sha256(json.dumps(_cache_contract,sort_keys=True).encode()).hexdigest()[:20]
cache=Path(CFG['cache_dir'])/CACHE_FINGERPRINT
cache.mkdir(parents=True,exist_ok=True)
print('Verified cache namespace:',CACHE_FINGERPRINT)
def causal_feature_normalize(frame,center,scale):
    """Train-only robust scaling and strictly causal 200-period rolling Z-score.
    Boolean data-presence fields stay unchanged, and normalization resets at gaps.
    """
    normalized=(frame[FEATURES]-center)/scale
    w=int(CFG['trailing_feature_window'])
    if w>1 and len(normalized)>1:
        cols=[c for c in FEATURES if c not in ('spread_present','volume_present')]
        ts=frame.index.to_numpy(dtype='datetime64[ns]')
        steps=np.diff(ts).astype('timedelta64[ns]').astype(np.int64)
        expected=int(steps.min())
        groups=np.r_[0,np.cumsum(steps!=expected)]
        vals=normalized[cols]
        rolled=vals.groupby(groups,sort=False).rolling(w,min_periods=min(w,48))
        mu=rolled.mean().reset_index(level=0,drop=True)
        sigma=rolled.std().reset_index(level=0,drop=True).clip(lower=.5)
        transformed=(vals-mu)/sigma
        normalized.loc[:,cols]=transformed.where(mu.notna(),vals)
    return normalized.clip(-12,12).fillna(0.).to_numpy('float32')

# Multi-timeframe bars are built from COMPLETE, CLOSED M5 bars only.
# Input M5 timestamps denote bar OPEN; a bar ending at t+5min is available only at that time.
TF_NAMES=list(CFG['mtf_timeframes'])
TF_FREQS=list(CFG['mtf_timeframes'].values())
TF_SECONDS=np.array([pd.Timedelta(x).total_seconds() for x in TF_FREQS],dtype=np.int64)
M5_SECONDS=int(CFG['bar_minutes']*60)
assert CFG['timestamp_semantics']=='open', 'MTF adapter requires M5 OPEN timestamps; verify source and change explicitly.'

def build_closed_timeframe_features(d,freq):
    freq_delta=pd.Timedelta(freq)
    ratio=int(freq_delta.total_seconds()/(CFG['bar_minutes']*60))
    assert ratio>=2 and ratio*(CFG['bar_minutes']*60)==freq_delta.total_seconds()
    # Index the input by true available end time, and aggregate each [start,end) interval.
    # Origin/closed/label are explicit, including midnight / UTC day boundaries.
    o=d[['open','high','low','close']].resample(freq,closed='left',label='right',origin='epoch').agg(
        {'open':'first','high':'max','low':'min','close':'last'})
    for field in ('volume','notional_usd','signed_notional_usd','buy_notional_usd','trade_count','dollar_event_count'):
        if field in d.columns:
            o[field]=d[field].resample(freq,closed='left',label='right',origin='epoch').sum(min_count=1)
    if 'dollar_event_imbalance' in d:
        o['dollar_event_imbalance']=d['dollar_event_imbalance'].resample(freq,closed='left',label='right',origin='epoch').mean()
    c=d['close'].resample(freq,closed='left',label='right',origin='epoch').count()
    valid=(c==ratio)
    # A full count is necessary but not sufficient: ensure the first and last M5 bar align.
    groups=d.index.to_series().resample(freq,closed='left',label='right',origin='epoch')
    first=groups.min();last=groups.max()
    valid &= (first==(c.index-freq_delta)) & (last==(c.index-pd.Timedelta(minutes=CFG['bar_minutes'])))
    o=o[valid].copy()
    # Require strictly non-null OHLC and prevent source-side aggregate lookahead.
    o=o.dropna(subset=['open','high','low','close'])
    if not o.empty: assert o.index.max()<=d.index.max()+pd.Timedelta(minutes=CFG['bar_minutes'])
    tf=causal_features(o)
    return tf

def tf_training_samples(tf,cut):
    x=tf.loc[(tf.index>=train_start)&(tf.index<cut),FEATURES]
    return x.iloc[::max(1,len(x)//25000)].copy() if len(x) else pd.DataFrame(columns=FEATURES)

H=CFG['horizons'];lb=CFG['lookback'];purge=CFG['purge_bars']
# Train-only robust statistics drawn from bounded deterministic per-asset samples
sampled=[];mtf_sampled={key:[] for key in TF_NAMES}
for path in paths:
    d=load_asset_file(path)
    d=d.loc[d.index>=train_start-pd.Timedelta(days=35)]
    f=causal_features(d)
    tr=f.loc[(f.index>=train_start)&(f.index<cuts[0]),FEATURES]
    if len(tr) and path.stem.replace('_M5','').upper() in CFG['pool_assets']:
        sampled.append(tr.iloc[::max(1,len(tr)//50000)].copy())
    for tf_name,tf_freq in CFG['mtf_timeframes'].items():
        tf=build_closed_timeframe_features(d,tf_freq)
        sample=tf_training_samples(tf,cuts[0])
        if len(sample) and path.stem.replace('_M5','').upper() in CFG['pool_assets']:
            mtf_sampled[tf_name].append(sample)
        del tf,sample
    del d,f,tr;gc.collect()
if not sampled:raise ValueError('No training rows for scaler')
scaler_samples=pd.concat(sampled,ignore_index=True);del sampled
med=scaler_samples.median().fillna(0.)
iqr=(scaler_samples.quantile(.75)-scaler_samples.quantile(.25)).replace(0,1.).fillna(1.)
del scaler_samples
mtf_scalers={}
for tf_name,rows in mtf_sampled.items():
    if not rows:raise ValueError(f'No training samples for {tf_name}')
    sample=pd.concat(rows,ignore_index=True)
    tf_med=sample.median().fillna(0.)
    tf_iqr=(sample.quantile(.75)-sample.quantile(.25)).replace(0,1.).fillna(1.)
    mtf_scalers[tf_name]=(tf_med,tf_iqr)
    del sample,rows
mtf_sampled.clear()
context_scalers={}
assets=[];fold_ids={k:[] for k in range(4)};counts={k:0 for k in range(4)}
for path in paths:
    asset=path.stem.replace('_M5','').upper()
    d=load_asset_file(path)
    d=d.loc[d.index>=train_start-pd.Timedelta(days=35)]
    f=causal_features(d);y=build_labels(d,H)
    # read-only source files never modified; cached arrays are local to runtime
    z=causal_feature_normalize(f,med,iqr)
    # Disk cache of timeframe features. Every row is indexed by its candle CLOSE time.
    for tf_name,tf_freq in CFG['mtf_timeframes'].items():
        tf=build_closed_timeframe_features(d,tf_freq)
        tf_med,tf_iqr=mtf_scalers[tf_name]
        tf_z=causal_feature_normalize(tf,tf_med,tf_iqr)
        tf_valid=np.isfinite(tf[['ret1','vol12','vol48']].to_numpy()).all(1).astype('bool')
        np.save(cache/f'{asset}_{tf_name}_features.npy',tf_z,allow_pickle=False)
        np.save(cache/f'{asset}_{tf_name}_times.npy',tf.index.to_numpy('datetime64[ns]'),allow_pickle=False)
        np.save(cache/f'{asset}_{tf_name}_valid.npy',tf_valid,allow_pickle=False)
        del tf,tf_z,tf_valid
    spread_valid=f['spread_present'].fillna(0).clip(0,1).to_numpy('float32')
    # Causal supervision, not ground-truth economic regimes; no future state used.
    regime_extra=np.column_stack([
        (np.log1p(np.maximum(f['spread_rel'].fillna(0).to_numpy(),0)*1e4)/np.log1p(50)).clip(0,1),
        (f['jump_proxy'].fillna(0).to_numpy()>5).astype('float32'),
        f['entropy24'].fillna(0).to_numpy().clip(0,1)]).astype('float32')
    regime_extra_mask=np.column_stack([spread_valid,np.ones(len(d),dtype='float32'),
                                       np.ones(len(d),dtype='float32')]).astype('float32')
    targets=np.column_stack([y[[f'ret_{h}' for h in H]].to_numpy('float32'),
      y[[f'dir_{h}' for h in H]].to_numpy('float32'),
      y[[f'mfe_{h}' for h in H]].to_numpy('float32'),
      y[[f'mae_{h}' for h in H]].to_numpy('float32'),
      y[[f'rv_{h}' for h in H]].to_numpy('float32'),
      y[['decision','regime_trend','regime_high_vol','regime_transition']].to_numpy('float32'),
      y[[f'pattern_{h}' for h in H]].to_numpy('float32'),
      y[[f'jump_{h}' for h in H]].to_numpy('float32'),
      y[[f'path_{h}' for h in H]].to_numpy('float32'),
      regime_extra,regime_extra_mask,
      y[[f'{kind}_{h}' for h in H for kind in ('false_breakout','reversal','continuation')]].to_numpy('float32'),
      y[[f'path_increment_{k}' for k in CFG['path_knots']]].to_numpy('float32'),
      y[[f'barrier_{h}' for h in H]].to_numpy('float32')])
    dates=f.index.to_numpy(dtype='datetime64[ns]');ns=len(dates)
    folds=np.searchsorted(np.asarray(cuts,dtype='datetime64[ns]'),dates,side='right')
    # exact indices where future labels, lookback and purge stay inside same fold
    # Decision/Barrier heads have separate masks: ambiguous OHLC same-bar touches must
    # NOT delete otherwise valid Forecast/SSL/Regime training examples.
    decision_valid=np.isfinite(targets[:,5*len(H)])
    barrier_valid=np.isfinite(targets[:,-len(H):])
    base_ok=np.isfinite(targets).copy()
    base_ok[:,5*len(H)]=True
    base_ok[:,-len(H):]=True
    valid=base_ok.all(1)
    np.save(cache/f'{asset}_decision_mask.npy',decision_valid.astype('bool'),allow_pickle=False)
    np.save(cache/f'{asset}_barrier_mask.npy',barrier_valid.astype('bool'),allow_pickle=False)
    targets[:,5*len(H)]=np.where(decision_valid,targets[:,5*len(H)],1.)
    targets[:,-len(H):]=np.where(barrier_valid,targets[:,-len(H):],1.)
    valid[:lb+purge]=False; valid[-max(H)-purge:]=False
    core=np.isfinite(f[['ret1','vol12','vol48']].to_numpy()).all(1)
    run=np.convolve((~core).astype('int32'),np.ones(lb,dtype='int32'),mode='full')[:ns]
    valid &= (run==0)
    # Source timestamps must be complete through the input lookback AND future max horizon.
    # This is O(N) and catches large gaps even when rolling features are finite.
    times_ns=dates.astype('datetime64[ns]')
    expected_ns=np.timedelta64(CFG['bar_minutes'],'m')
    connected_past=np.zeros(ns,dtype=bool)
    connected_past[lb-1:]=(times_ns[lb-1:]-times_ns[:ns-lb+1])==(lb-1)*expected_ns
    connected_future=np.zeros(ns,dtype=bool)
    hh=max(H)
    connected_future[:-hh]=(times_ns[hh:]-times_ns[:-hh])==hh*expected_ns
    valid &= connected_past & connected_future
    indexes=np.where(valid)[0]
    indexes=indexes[(indexes+purge<ns)&(indexes-purge>=0)]
    indexes=indexes[dates[indexes]>=train_start.tz_localize(None).to_datetime64()]
    indexes=indexes[(folds[indexes]==folds[indexes-purge])&
      (folds[indexes]==folds[indexes+purge])&
      (folds[indexes]==folds[indexes-lb+1])&
      (folds[indexes]==folds[indexes+max(H)])]
    # Historical stress is selected using only CAUSAL realized volatility at the decision bar.
    train_vol=f.loc[(f.index>=train_start)&(f.index<train_end),'vol48'].to_numpy(dtype=np.float64)
    train_vol=train_vol[np.isfinite(train_vol)]
    stress_threshold=float(np.quantile(train_vol,CFG['stress_vol_quantile'])) if len(train_vol)>100 else float('inf')
    stress=(f['vol48'].fillna(0).to_numpy(dtype=np.float64)>=stress_threshold).astype('uint8')
    np.save(cache/f'{asset}_causal_stress.npy',stress,allow_pickle=False)
    aid=len(assets)
    np.save(cache/f'{asset}_features.npy',z,allow_pickle=False)
    np.save(cache/f'{asset}_targets.npy',targets.astype('float32'),allow_pickle=False)
    np.save(cache/f'{asset}_times.npy',dates,allow_pickle=False)
    assets.append(asset)
    for k in range(4):
        chosen=(indexes[folds[indexes]==k].astype('int32') if asset in CFG['pool_assets']
                else np.empty(0,dtype='int32'))
        np.save(cache/f'{asset}_fold{k}.npy',chosen,allow_pickle=False)
        counts[k]+=len(chosen)
    if asset in CFG['pool_assets']:
        asset_counts={k:int(((folds[indexes]==k)).sum()) for k in range(4)}
        if asset_counts[0]<CFG['min_train_sequences']:
            raise ValueError(f'Pooled training would omit {asset}: {asset_counts}')
        if asset==CFG['target_asset'] and any(asset_counts[k]==0 for k in (1,2,3)):
            raise ValueError(f'BTC target must have valid validation/calibration/OOS: {asset_counts}')
    print(asset,'bars',ns,'samples per fold',[(k,int((folds[indexes]==k).sum())) for k in range(4)])
    del d,f,y,z,targets,dates,folds,indexes;gc.collect()
if counts[0]<CFG['min_train_sequences'] or any(counts[k]==0 for k in (1,2,3)):
    raise ValueError(f'Insufficient valid samples per split: {counts}')
print('Counts:',counts)

# Precompute causal as-of indices for graph neighbors once, rather than repeating
# Avoid repeated cross-asset searchsorted calls across pooled epochs.
# Each row is an M5 OPEN time; neighbor indices always refer to timestamps <= that time.
# Save as numpy .npy memmaps (no full N_assets x N_rows tensor in RAM).
if CFG.get('graph_index_cache',True):
    all_times=[np.load(cache/f'{a}_times.npy',mmap_mode='r') for a in assets]
    graph_cache_info={}
    for aid,asset in enumerate(assets):
        outfile=cache/f'{asset}_cross_asset_index.npy'
        rows=len(all_times[aid])
        arr=np.lib.format.open_memmap(outfile,mode='w+',dtype=np.int32,shape=(rows,len(assets)))
        for oid in range(len(assets)):
            if oid==aid:
                arr[:,oid]=-1
                continue
            other=all_times[oid]
            # Chunked to avoid creating one giant transient index array.
            for left in range(0,rows,200000):
                right=min(rows,left+200000)
                idx=np.searchsorted(other,all_times[aid][left:right],side='right').astype(np.int64)-1
                if len(other)>=2**31:raise ValueError('graph index exceeds int32 capacity')
                arr[left:right,oid]=idx.astype(np.int32)
        arr.flush();del arr
        graph_cache_info[asset]={'rows':rows,'neighbors':len(assets)-1,'path':str(outfile)}
    del all_times
    print('Graph alignment cache ready:',graph_cache_info)

class LazyFold(Dataset):
    def __init__(self,k,selected_assets=None,validation_role="selection"):
        self.k=k;self.selected_assets=set(selected_assets or CFG['pool_assets']);self.loc=[];self.X=[];self.Y=[];self.T=[];self.ids=[];self.M=[];self.MT=[];self.MV=[];self.GI=[];self.DM=[];self.BM=[]
        for aid,asset in enumerate(assets):
            rows=np.load(cache/f'{asset}_fold{k}.npy',mmap_mode='r')
            if k==1 and validation_role in ('selection','report'):
                # Maintain a distinct untouched diagnostic holdout inside validation.
                _ts=np.load(cache/f'{asset}_times.npy',mmap_mode='r')
                _cut=(cal_start-pd.DateOffset(months=CFG['validation_report_months'])).tz_localize(None).to_datetime64()
                _purge=np.timedelta64(max(H)+CFG['purge_bars'],'m')*CFG['bar_minutes']
                _chosen=np.asarray(rows,dtype=np.int64)
                _when=np.asarray(_ts[_chosen])
                keep=(_when<_cut-_purge) if validation_role=='selection' else (_when>=_cut)
                rows=_chosen[keep]
            self.ids.append(rows)
            self.GI.append(np.load(cache/f'{asset}_cross_asset_index.npy',mmap_mode='r') if CFG.get('graph_index_cache',True) else None)
            self.X.append(np.load(cache/f'{asset}_features.npy',mmap_mode='r'))
            self.Y.append(np.load(cache/f'{asset}_targets.npy',mmap_mode='r'))
            self.DM.append(np.load(cache/f'{asset}_decision_mask.npy',mmap_mode='r'))
            self.BM.append(np.load(cache/f'{asset}_barrier_mask.npy',mmap_mode='r'))
            self.T.append(np.load(cache/f'{asset}_times.npy',mmap_mode='r'))
            self.M.append([np.load(cache/f'{asset}_{tf}_features.npy',mmap_mode='r') for tf in TF_NAMES])
            self.MT.append([np.load(cache/f'{asset}_{tf}_times.npy',mmap_mode='r') for tf in TF_NAMES])
            self.MV.append([np.load(cache/f'{asset}_{tf}_valid.npy',mmap_mode='r') for tf in TF_NAMES])
            self.loc.append((aid,len(rows) if asset in self.selected_assets else 0))
        self.ends=np.cumsum([n for _,n in self.loc]);self.count=int(self.ends[-1])
    def __len__(self):return self.count
    def multi_timeframe(self,aid,i):
        # M5 bar i becomes observable at its CLOSE; no unfinished M15/H1/H4/D1 bar.
        asof=self.T[aid][i] + np.timedelta64(CFG['bar_minutes'],'m')
        window=int(CFG['mtf_window']);Fdim=len(FEATURES)
        out=np.zeros((len(TF_NAMES),window,Fdim),dtype='float32')
        mask=np.zeros((len(TF_NAMES),window),dtype=bool)
        for j,tf_name in enumerate(TF_NAMES):
            times=self.MT[aid][j];features=self.M[aid][j];quality=self.MV[aid][j]
            stop=int(np.searchsorted(times,asof,side='right'))
            if stop==0:continue
            # Inactive markets cannot silently carry stale multi-timeframe context.
            age=(asof-times[stop-1])/np.timedelta64(1,'s')
            if age>TF_SECONDS[j]*2:continue
            start=max(0,stop-window);n=stop-start
            out[j,window-n:]=features[start:stop]
            mask[j,window-n:]=quality[start:stop]
            assert np.all(times[start:stop]<=asof),'MTF feature leakage'
        return torch.from_numpy(out),torch.from_numpy(mask)
    def __getitem__(self,k):
        pos=int(np.searchsorted(self.ends,k,side='right'));aid=self.loc[pos][0]
        prior=0 if pos==0 else int(self.ends[pos-1]);i=int(self.ids[aid][k-prior])
        x=np.array(self.X[aid][i-lb+1:i+1],dtype='float32',copy=True)
        y=np.array(self.Y[aid][i],dtype='float32',copy=True)
        # Dynamic graph snapshots at multiple PAST anchor times, never future timestamps.
        hp=CFG['graph_history_points'];stride=CFG['graph_history_stride']
        anchors=np.array([self.T[aid][i-(hp-1-j)*stride] for j in range(hp)])
        neighbors=[];available=[];couplings=[]
        center_ret=x[-1-(hp-1)*stride::stride,0][-hp:]  # causal own past changes, approximate alignment
        for oid in range(len(self.X)):
            if oid==aid:continue
            other_t=self.T[oid];other_x=self.X[oid]
            ix=(np.asarray(self.GI[aid][i-(hp-1)*stride:i+1:stride,oid],dtype=np.int64)
                if self.GI[aid] is not None else np.searchsorted(other_t,anchors,side='right')-1)
            valid=(ix>=0)
            age=np.full(hp,np.timedelta64(10**9,'m'),dtype='timedelta64[m]')
            age[valid]=anchors[valid]-other_t[ix[valid]]
            valid &= age<=np.timedelta64(15,'m')
            hist=np.zeros((hp,len(FEATURES)),dtype='float32')
            hist[valid]=other_x[ix[valid]]
            neighbors.append(hist);available.append(valid)
            if valid.sum()>=6 and np.std(center_ret[valid])>1e-8 and np.std(hist[valid,0])>1e-8:
                corr=np.corrcoef(center_ret[valid],hist[valid,0])[0,1]
                if np.isfinite(corr):couplings.append(abs(corr))
        neighbors=np.stack(neighbors,axis=0) if neighbors else np.zeros((0,hp,len(FEATURES)),dtype='float32')
        mask=np.stack(available,axis=0) if available else np.zeros((0,hp),dtype=bool)
        start=8*len(H)+4
        regime_tail=y[start:start+3]
        regime_mask=y[start+3:start+6]
        pat_start=start+6
        pattern_detail=y[pat_start:pat_start+len(H)*3].reshape(len(H),3)
        path_start=pat_start+len(H)*3
        future_path=y[path_start:path_start+len(CFG['path_knots'])]
        barrier_targets=y[path_start+len(CFG['path_knots']):path_start+len(CFG['path_knots'])+len(H)]
        coupling=float(np.mean(couplings)) if couplings else 0.
        regime_full=np.concatenate([y[5*len(H)+1:5*len(H)+4],regime_tail,[coupling]]).astype('float32')
        regime_full_mask=np.concatenate([np.ones(3,dtype='float32'),regime_mask,[float(bool(couplings))]]).astype('float32')
        mtf,mtf_mask=self.multi_timeframe(aid,i)
        return (torch.from_numpy(x),torch.from_numpy(y[:len(H)]),
          torch.from_numpy(y[len(H):2*len(H)].astype('int64')),
          torch.from_numpy(y[2*len(H):3*len(H)]),
          torch.from_numpy(y[3*len(H):4*len(H)]),
          torch.from_numpy(y[4*len(H):5*len(H)]),
          torch.tensor(int(y[5*len(H)]),dtype=torch.long),
          torch.from_numpy(regime_full),
          torch.from_numpy(y[5*len(H)+4:6*len(H)+4]),
          torch.from_numpy(y[6*len(H)+4:7*len(H)+4]),
          torch.from_numpy(y[7*len(H)+4:8*len(H)+4]),
          torch.from_numpy(neighbors),torch.from_numpy(mask),
          torch.from_numpy(regime_full_mask),torch.from_numpy(pattern_detail),
          torch.from_numpy(future_path),torch.from_numpy(barrier_targets.astype('int64')),mtf,mtf_mask,
          torch.tensor(bool(self.DM[aid][i])),torch.from_numpy(np.asarray(self.BM[aid][i],dtype=bool).copy()))
    def ref(self,k):
        pos=int(np.searchsorted(self.ends,k,side='right'));aid=self.loc[pos][0]
        prior=0 if pos==0 else int(self.ends[pos-1]);i=int(self.ids[aid][k-prior])
        return assets[aid],str(pd.Timestamp(self.T[aid][i],tz='UTC') + pd.Timedelta(minutes=CFG['bar_minutes']) if CFG['timestamp_semantics']=='open' else pd.Timestamp(self.T[aid][i],tz='UTC'))
def training_sample_weights(selected_assets=None):
    """Train-only exponential recency weights with a floor and bounded rare stress boost.
    No future labels, validation, calibration or sealed OOS rows enter these weights.
    """
    parts=[];audit=[]
    for asset in assets:
        if asset not in set(selected_assets or CFG['pool_assets']):continue
        ids=np.load(cache/f'{asset}_fold0.npy',mmap_mode='r')
        timestamps=np.load(cache/f'{asset}_times.npy',mmap_mode='r')
        stress=np.load(cache/f'{asset}_causal_stress.npy',mmap_mode='r')
        if not len(ids):raise RuntimeError(f'No train examples: {asset}')
        t=np.asarray(timestamps[ids],dtype='datetime64[ns]')
        age_days=np.maximum((train_end.tz_localize(None).to_datetime64()-t)/np.timedelta64(1,'D'),0.)
        recent=np.exp2(-age_days/float(CFG['recent_half_life_days']))
        weight=float(CFG['recent_uniform_floor'])+(1.-float(CFG['recent_uniform_floor']))*recent
        flags=np.asarray(stress[ids],dtype=np.float64)
        weight*=np.where(flags>0.,min(float(CFG['stress_boost']),float(CFG['stress_cap'])),1.)
        # Equal probability mass per training asset, without removing recency/stress ordering within it.
        # WeightedRandomSampler samples independently with replacement per epoch.
        weight=weight/max(weight.sum(),1e-12)
        parts.append(weight.astype('float64'))
        audit.append({'asset':asset,'train_rows':int(len(ids)),'stress_rows':int(flags.sum()),
            'weight_min':float(weight.min()),'weight_max':float(weight.max()),
            'recent_365d_rows':int((age_days<=365.).sum())})
    weights=np.concatenate(parts)
    if not np.isfinite(weights).all() or np.any(weights<=0):raise RuntimeError('Invalid training weights')
    print('TRAIN-ONLY recency/stress sampling:',json.dumps(audit))
    return torch.from_numpy(weights)

class DeterministicEpochSampler(torch.utils.data.Sampler):
    """Independent seeded weighted epochs; resume at exact optimizer-step boundary."""
    def __init__(self,weights,num_samples):
        self.weights=weights.double();self.total=int(num_samples)
        self.epoch_seed=47;self.start=0
    def set_epoch(self,phase,stage,epoch,completed_steps=0):
        token=f"{SEED}:{phase}:{stage}:{epoch}"
        self.epoch_seed=int(hashlib.sha256(token.encode()).hexdigest()[:15],16)
        self.start=int(completed_steps)*int(CFG['batch_size'])
        if not 0<=self.start<=self.total:raise RuntimeError('Invalid resume batch offset')
    def __iter__(self):
        g=torch.Generator().manual_seed(self.epoch_seed)
        seq=torch.multinomial(self.weights,self.total,replacement=True,generator=g)
        return iter(seq[self.start:].tolist())
    def __len__(self):return max(0,self.total-self.start)

_FOLD_LOADERS={}
def dl(k,shuffle=False,selected_assets=None,validation_role="selection"):
    selected_assets=tuple(selected_assets or CFG['pool_assets'])
    key=(int(k),bool(shuffle),selected_assets,validation_role)
    if key not in _FOLD_LOADERS:
        workers=int(CFG['loader_workers'])
        sampler=(DeterministicEpochSampler(training_sample_weights(selected_assets),
                 num_samples=len(LazyFold(0,selected_assets))) if k==0 and shuffle else None)
        kwargs={'batch_size':CFG['batch_size'],'shuffle':bool(shuffle and sampler is None),'sampler':sampler,
                'num_workers':workers,'pin_memory':DEVICE.type=='cuda',
                'generator':torch.Generator().manual_seed(SEED+int(k))}
        if workers>0:
            kwargs.update(persistent_workers=bool(CFG['persistent_workers']),
                          prefetch_factor=int(CFG['prefetch_factor']))
        _FOLD_LOADERS[key]=DataLoader(LazyFold(k,selected_assets,validation_role),**kwargs)
    return _FOLD_LOADERS[key]

def move_batch(pack):
    return tuple(z.to(DEVICE,non_blocking=DEVICE.type=='cuda') for z in pack)
# Deterministic target scales from TRAIN sampled across assets without X-window materialization
trvals=[]
for asset in CFG['pool_assets']:
    t=np.load(cache/f'{asset}_targets.npy',mmap_mode='r');ids=np.load(cache/f'{asset}_fold0.npy',mmap_mode='r')
    subset=np.asarray(ids[::max(1,len(ids)//10000)],dtype=int)
    if len(subset):trvals.append(np.array(t[subset],copy=True))
trvals=np.concatenate(trvals)
ret_scale=max(float(np.median(np.abs(trvals[:,len(H)-1]))),1e-5)
rv_scale=max(float(np.median(trvals[:,5*len(H)-1])),1e-8)
del trvals
print('Train-only scale:',ret_scale,rv_scale)

# Exhaustive as-of audit for every supervised fold row, not three sampled positions.
# No higher-timeframe value later than the decision timestamp can enter the model.
for _asset in assets:
    _base=np.load(cache/f'{_asset}_times.npy',mmap_mode='r')
    for _k in range(4):
        _ids=np.load(cache/f'{_asset}_fold{_k}.npy',mmap_mode='r')
        if not len(_ids):continue
        _asofs=np.asarray(_base[_ids])+np.timedelta64(CFG['bar_minutes'],'m')
        for _tf in TF_NAMES:
            _tf_times=np.load(cache/f'{_asset}_{_tf}_times.npy',mmap_mode='r')
            _stops=np.searchsorted(_tf_times,_asofs,side='right')
            _has=_stops>0
            if _has.any():
                assert np.all(_tf_times[_stops[_has]-1]<=_asofs[_has]),(_asset,_k,_tf)
print('Exhaustive MTF as-of index audit PASS')
# MTF invariant test on cached datasets: every historical higher-timeframe candle closes <= asof.
for _fold in range(4):
    _ds=LazyFold(_fold,CFG['pool_assets'])
    for _q in sorted(set([0,len(_ds)//2,len(_ds)-1])):
        _aid=int(np.searchsorted(_ds.ends,_q,side='right'))
        _prior=0 if _aid==0 else int(_ds.ends[_aid-1]);_i=int(_ds.ids[_aid][_q-_prior])
        _asof=_ds.T[_aid][_i]+np.timedelta64(CFG['bar_minutes'],'m')
        for _j in range(len(TF_NAMES)):
            _stop=int(np.searchsorted(_ds.MT[_aid][_j],_asof,side='right'))
            assert _stop==0 or _ds.MT[_aid][_j][_stop-1]<=_asof
print('MTF closed-candle invariant OK')
assert len(dl(1,False,[CFG['target_asset']],validation_role='report').dataset)>0, 'Empty independent validation report fold'


## 3. Quant Transformer + Regime-guided sparse MoE

The model has no asset-ID embeddings. Relative price statistics allow an **unseen** asset to be encoded, but do not establish alpha on that asset. Eight experts with top-2 routing contribute a shared representation. Three supervised causal-regime teacher channels make the regime output interpretable. Forecasting heads produce return quantiles, direction classes, future realized variance, excursion and barrier outcomes. The final decision head is separately trained on research barrier outcomes, **not** copied from direction labels. No portfolio allocation or broker execution code.
**Single-asset mode:** Cross-asset GNN has no other asset neighbors; Stage C must not be interpreted as learning cross-asset relationships.


In [ ]:
# Full probabilistic architecture — causal sequence + evolving graph + direct bypass + stochastic paths
class CausalDiagonalStateSpace(nn.Module):
    """Learnable stable diagonal linear recurrence using parallel causal convolution.
    Kernel k_j=(1-alpha)*alpha**j; all outputs at t use inputs <=t only.
    """
    def __init__(self,dim):
        super().__init__()
        self.decay_logits=nn.Parameter(torch.linspace(.5,5.,dim))
        self.in_projection=nn.Linear(dim,dim)
        self.out_gate=nn.Linear(dim,dim)
        self.norm=nn.LayerNorm(dim)
    def forward(self,h):
        B,T,D=h.shape
        alpha=torch.sigmoid(self.decay_logits).clamp(.05,.999)
        lags=torch.arange(T,device=h.device,dtype=h.dtype)
        impulse=(1-alpha[:,None])*alpha[:,None].pow(lags[None,:])
        states=F.conv1d(self.in_projection(h).transpose(1,2),
                        impulse.flip(-1).unsqueeze(1),padding=T-1,groups=D)[...,:T]
        return self.norm(states.transpose(1,2)*torch.sigmoid(self.out_gate(h)))

class DynamicTemporalGraph(nn.Module):
    def __init__(self,dim,heads=6,layers=2):
        super().__init__()
        self.layers=nn.ModuleList([nn.MultiheadAttention(dim,heads,batch_first=True) for _ in range(layers)])
        self.norms=nn.ModuleList([nn.LayerNorm(dim) for _ in range(layers)])
        self.temporal=nn.GRU(dim,dim,batch_first=True)
        self.fuse=nn.Sequential(nn.Linear(dim*2,dim),nn.GELU(),nn.LayerNorm(dim))
    def forward(self,center,center_history,nodes,mask):
        # [B,N,T,D] per-time dynamic neighbor attention; every snapshot is <= center anchor.
        B,N,T,D=nodes.shape
        assert center_history.shape==(B,T,D), 'Graph center states must align with graph snapshots'
        center_repeat=center_history.reshape(B*T,1,D)
        z=nodes.permute(0,2,1,3).reshape(B*T,N,D)
        valid=mask.permute(0,2,1).reshape(B*T,N)
        tokens=torch.cat([center_repeat,z],dim=1)
        padding=torch.cat([torch.zeros((B*T,1),dtype=torch.bool,device=center.device),~valid],dim=1)
        for attn,norm in zip(self.layers,self.norms):
            upd,_=attn(tokens,tokens,tokens,key_padding_mask=padding,need_weights=False)
            tokens=norm(tokens+upd)
        seq=tokens[:,0].reshape(B,T,D)
        state=self.temporal(seq)[0][:,-1]
        any_neighbor=mask.any((1,2))
        fused=self.fuse(torch.cat([center,state],dim=-1))
        return torch.where(any_neighbor[:,None],fused,center)

class MultiTimeframeFusion(nn.Module):
    """Independent causal M15/H1/H4/D1 encoders plus mask-aware attention to M5.
    Historical windows end at the last CLOSED bar available at the M5 decision time.
    Missing timeframes have zero attention probability, not merely zero-filled inputs.
    """
    def __init__(self,features,hidden,heads,layers):
        super().__init__()
        self.encoders=nn.ModuleList([nn.TransformerEncoder(
            nn.TransformerEncoderLayer(hidden,heads,hidden*2,dropout=.1,batch_first=True,norm_first=True),
            layers,enable_nested_tensor=False) for _ in TF_NAMES])
        self.input=nn.ModuleList([nn.Sequential(nn.Linear(features,hidden),nn.LayerNorm(hidden)) for _ in TF_NAMES])
        self.tokens=nn.Parameter(torch.randn(len(TF_NAMES),hidden)*.02)
        self.query=nn.Linear(hidden,hidden)
        self.key=nn.Linear(hidden,hidden)
        self.value=nn.Linear(hidden,hidden)
        self.output=nn.Sequential(nn.Linear(hidden*2,hidden),nn.GELU(),nn.LayerNorm(hidden))
        self.norm=nn.LayerNorm(hidden)
        self.scale=hidden**-.5
    def forward(self,query,mtf,mtf_mask):
        B,K,W,Fdim=mtf.shape
        assert K==len(TF_NAMES) and mtf_mask.shape==(B,K,W)
        tokens=[];present=[]
        causal=torch.triu(torch.ones(W,W,device=mtf.device,dtype=torch.bool),diagonal=1)
        for j in range(K):
            good=mtf_mask[:,j,:].bool()
            # all-zero samples must still produce finite outputs; output suppressed by present mask.
            h=self.input[j](mtf[:,j])+self.tokens[j]
            h=self.encoders[j](h,mask=causal)
            idx=good.long().sum(1).sub(1).clamp(min=0)
            # Padding on left: the last candle is always W-1 if this timeframe exists.
            token=h[:,-1]
            valid=good.any(1)
            tokens.append(token);present.append(valid)
        t=torch.stack(tokens,dim=1)
        available=torch.stack(present,dim=1)
        scores=(self.query(query)[:,None,:]*self.key(t)).sum(-1)*self.scale
        scores=scores.masked_fill(~available,-1e9)
        alpha=F.softmax(scores,dim=-1)*available.float()
        alpha=alpha/alpha.sum(-1,keepdim=True).clamp(min=1e-8)
        fused=(alpha.unsqueeze(-1)*self.value(t)).sum(1)
        blended=self.output(torch.cat([query,fused],dim=-1))
        return torch.where(available.any(1,keepdim=True),blended,query),alpha

class UniversalQuantModel(nn.Module):
    def __init__(self,features,hidden=192,nexperts=8,topk=2,nh=4,
                 transformer_layers=5,attention_heads=6,gru_layers=2,graph_layers=2,expert_width=512):
        super().__init__()
        self.nh=nh;self.topk=topk;self.npath=len(CFG['path_knots']);self.rank=CFG['path_rank']
        self.input=nn.Sequential(nn.Linear(features,hidden),nn.LayerNorm(hidden))
        self.conv=nn.Conv1d(hidden,hidden,5,padding=4,groups=hidden)
        self.encoder=nn.TransformerEncoder(nn.TransformerEncoderLayer(hidden,attention_heads,hidden*4,
            dropout=.1,batch_first=True,norm_first=True),transformer_layers,enable_nested_tensor=False)
        self.longrange=nn.GRU(hidden,hidden,num_layers=gru_layers,batch_first=True)
        self.ssm=CausalDiagonalStateSpace(hidden)
        self.graph_input=nn.Sequential(nn.Linear(features,hidden),nn.GELU(),nn.LayerNorm(hidden))
        self.graph=DynamicTemporalGraph(hidden,attention_heads,graph_layers)
        self.graph_fusion=nn.Sequential(nn.Linear(hidden*4,hidden),nn.GELU(),nn.LayerNorm(hidden))
        self.mtf=MultiTimeframeFusion(features,hidden,attention_heads,CFG['mtf_encoder_layers'])
        self.ssl_predict=nn.Linear(hidden,features)
        self.regime=nn.Sequential(nn.Linear(hidden,hidden),nn.GELU(),nn.Linear(hidden,7))
        self.router=nn.Sequential(nn.Linear(hidden+7,hidden),nn.GELU(),nn.Linear(hidden,nexperts))
        self.experts=nn.ModuleList([nn.Sequential(nn.Linear(hidden,expert_width),nn.GELU(),
                           nn.Linear(expert_width,expert_width),nn.GELU(),nn.Linear(expert_width,hidden))
                           for _ in range(nexperts)])
        # Direct connections from original market context, graph and regime; no mandatory MoE bottleneck.
        self.forecast_fusion=nn.Sequential(nn.Linear(hidden*3+7,hidden),nn.GELU(),nn.LayerNorm(hidden))
        self.direction=nn.Linear(hidden,nh*3)
        self.quantiles=nn.Linear(hidden,nh*3)
        self.rv=nn.Linear(hidden,nh)
        self.excursions=nn.Linear(hidden,nh*2)
        self.barrier=nn.Linear(hidden,nh*3)
        self.pattern=nn.Linear(hidden,nh)
        self.pattern_detail=nn.Linear(hidden,nh*3)
        self.jump=nn.Linear(hidden,nh)
        self.path=nn.Linear(hidden,nh) # legacy coarse shape, retained for continuity
        self.path_distribution=nn.Linear(hidden,self.npath*(2+self.rank))
        self.decision_quantiles=nn.Linear(hidden+nh*3,3*3)  # action utility q10/q50/q90
        self.decision=nn.Sequential(nn.Linear(hidden+nh*3+9,hidden),nn.GELU(),nn.Linear(hidden,3))
    def forward(self,x,neighbors=None,neighbor_mask=None,use_experts=True,ablated_expert=None,
                use_graph=True,mtf=None,mtf_mask=None,use_mtf=True):
        B,T,_=x.shape
        h=self.input(x)
        h=h+self.conv(h.transpose(1,2))[...,:T].transpose(1,2)
        causal=torch.triu(torch.ones(T,T,dtype=torch.bool,device=x.device),diagonal=1)
        temporal_states=self.encoder(h,mask=causal)
        temporal=temporal_states[:,-1]
        recurrent=self.longrange(h)[0][:,-1]
        state_space=self.ssm(h)[:,-1]
        graph_repr=temporal
        if use_graph and neighbors is not None and neighbors.shape[1]>0:
            assert neighbors.ndim==4 and neighbor_mask.ndim==3
            nodes=self.graph_input(neighbors)
            gp=neighbors.shape[2];stride=CFG['graph_history_stride']
            history=temporal_states[:,-1-(gp-1)*stride::stride]
            if history.shape[1]!=gp:raise ValueError('Lookback shorter than graph history span')
            graph_repr=self.graph(temporal,history,nodes,neighbor_mask.bool())
        z0=self.graph_fusion(torch.cat([temporal,recurrent,state_space,graph_repr],-1))
        if use_mtf and mtf is not None:
            assert mtf_mask is not None
            z0,mtf_attention=self.mtf(z0,mtf,mtf_mask)
        else:
            mtf_attention=torch.zeros(B,len(TF_NAMES),device=x.device,dtype=x.dtype)
        regime=self.regime(z0)
        route=self.router(torch.cat([z0,torch.sigmoid(regime)],-1))
        top=route.topk(min(self.topk,route.shape[-1]),dim=-1)
        gate=torch.zeros_like(route).scatter(1,top.indices,F.softmax(top.values.float(),dim=-1).to(route.dtype))
        if ablated_expert is not None:
            gate=gate.clone();gate[:,int(ablated_expert)]=0.
            gate=gate/gate.sum(1,keepdim=True).clamp(min=1e-8)
        # Sparse dispatch: only run experts selected for each sample.
        mixture=torch.zeros_like(z0)
        if use_experts:
            for j,expert in enumerate(self.experts):
                selected=torch.nonzero(gate[:,j]>0,as_tuple=True)[0]
                if selected.numel():
                    expert_contribution=expert(z0.index_select(0,selected))*gate[selected,j,None]
                    # Under BF16 autocast, expert output may differ from FP32 mixture dtype.
                    # index_add requires matching dtypes; this conversion retains gradients.
                    mixture=mixture.index_add(0,selected,expert_contribution.to(dtype=mixture.dtype))
        z=z0+mixture
        ctx=z+self.forecast_fusion(torch.cat([z,z0,graph_repr,torch.sigmoid(regime)],dim=-1))
        direction=self.direction(ctx).reshape(B,self.nh,3)
        raw=self.quantiles(ctx).reshape(B,self.nh,3)
        median=raw[...,1]*ret_scale
        quantiles=torch.stack([median-F.softplus(raw[...,0])*ret_scale,median,
                               median+F.softplus(raw[...,2])*ret_scale],dim=-1)
        raw_path=self.path_distribution(ctx).reshape(B,self.npath,2+self.rank)
        path_mu=raw_path[...,0]*ret_scale
        path_diag=(F.softplus(raw_path[...,1]).clamp(max=10.)+.03)*ret_scale
        path_factors=torch.tanh(raw_path[...,2:])*ret_scale
        direction_probs=F.softmax(direction,dim=-1).reshape(B,-1)
        dq=self.decision_quantiles(torch.cat([ctx,direction_probs],dim=-1)).reshape(B,3,3)
        utility_median=dq[...,1]
        action_quantiles=torch.stack([utility_median-F.softplus(dq[...,0]),utility_median,
                                    utility_median+F.softplus(dq[...,2])],dim=-1)*ret_scale
        decision=self.decision(torch.cat([ctx,direction_probs,action_quantiles.reshape(B,-1)/ret_scale],-1))
        return {'direction':direction,'quantiles':quantiles,'rv':F.softplus(self.rv(ctx))*rv_scale,
                'excursions':self.excursions(ctx).reshape(-1,self.nh,2)*ret_scale,
                'barrier':self.barrier(ctx).reshape(-1,self.nh,3),'pattern':self.pattern(ctx),
                'pattern_detail':self.pattern_detail(ctx).reshape(-1,self.nh,3),
                'jump':self.jump(ctx),'path':self.path(ctx),
                'path_mu':path_mu,'path_diag':path_diag,'path_factors':path_factors,
                'action_quantiles':action_quantiles,'decision':decision,'regime':regime,
                'ssl_prediction':self.ssl_predict(z0),'gate':gate,
                'router_probs':F.softmax(route,dim=-1),'dispatch':(gate>0).float(),
                'mtf_attention':mtf_attention}

def path_distribution_nll(o,target):
    # NVIDIA L4: CUDA autocast may downcast the internal matrix multiply to BF16.
    # LowRankMultivariateNormal then calls cuSOLVER Cholesky, which rejects BF16.
    # Disable autocast for the ENTIRE distribution construction/log_prob,
    # not just the input casts; gradients still flow back to the BF16 model.
    with torch.autocast(device_type=o['path_mu'].device.type, enabled=False):
        scale=float(ret_scale)
        mu=o['path_mu'].float()/scale
        factors=o['path_factors'].float()/scale
        diagonal=(o['path_diag'].float()/scale).square().clamp(min=1e-5)
        dist=torch.distributions.LowRankMultivariateNormal(
            loc=mu, cov_factor=factors, cov_diag=diagonal)
        nll=-dist.log_prob(target.float()/scale).mean()
    return nll

@torch.no_grad()
def path_forecast_quantiles(o):
    # True path distribution: cumulative correlated Gaussian increments at future knots.
    mu=o['path_mu'].cumsum(-1)
    diag_var=o['path_diag'].square().cumsum(-1)
    factor_sum=o['path_factors'].cumsum(1)
    var=diag_var+factor_sum.square().sum(-1)
    sd=var.clamp(min=1e-15).sqrt()
    return torch.stack([mu-1.2815515655*sd,mu,mu+1.2815515655*sd],dim=-1)

@torch.no_grad()
def sample_correlated_price_paths(o,starting_price,scenarios=64):
    # Draw correlated log-return increments in FP32 even if caller uses CUDA BF16 autocast.
    with torch.autocast(device_type=o['path_mu'].device.type, enabled=False):
        dist=torch.distributions.LowRankMultivariateNormal(
            loc=o['path_mu'].float(),cov_factor=o['path_factors'].float(),
            cov_diag=o['path_diag'].float().square().clamp(min=1e-14))
        increments=dist.sample((int(scenarios),)) # [scenarios, batch, path_knots]
        return float(starting_price)*increments.cumsum(-1).exp()

model=UniversalQuantModel(len(FEATURES),CFG['hidden'],CFG['experts'],CFG['top_k'],len(H),
    CFG['transformer_layers'],CFG['attention_heads'],CFG['gru_layers'],CFG['graph_layers'],CFG['expert_width']).to(DEVICE)
PARAMETERS=sum(p.numel() for p in model.parameters())
print('FULL probabilistic trainable parameters:',PARAMETERS)


In [ ]:
# Objective and eight-stage progressive optimizer: SSL → regime → graph → forecast → path → experts → decision → joint.
def pinball(pred,y):
    tau=torch.tensor([.1,.5,.9],dtype=pred.dtype,device=pred.device)
    error=y.unsqueeze(-1)-pred
    return torch.maximum(tau*error,(tau-1)*error).mean()

def cost_aware_utilities(r,m,n):
    # Research-only, future teacher; NEVER available to model during inference.
    terminal=r[:,-1]
    long=terminal-CFG['reference_cost_bps']*1e-4-CFG['research_adverse_penalty']*(-n[:,-1]).clamp(min=0)
    short=-terminal-CFG['reference_cost_bps']*1e-4-CFG['research_adverse_penalty']*m[:,-1].clamp(min=0)
    return torch.stack([short,torch.zeros_like(terminal),long],dim=-1)

def regime_loss(o,b):
    target=b[7];mask=b[13]
    assert target.shape[1]==7 and mask.shape==target.shape
    err=F.binary_cross_entropy_with_logits(o['regime'],target,reduction='none')
    return (err*mask).sum()/mask.sum().clamp(min=1)

def masked_mean(loss,mask):
    mask=mask.to(dtype=loss.dtype)
    return (loss*mask).sum()/mask.sum().clamp(min=1)

def masked_barrier_loss(o,b):
    B=len(b[0]);nh=len(H)
    logits=o['barrier'].float().reshape(-1,3)
    losses=F.cross_entropy(logits,b[16].long().reshape(-1),reduction='none').reshape(B,nh)
    return masked_mean(losses,b[20])

def masked_decision_loss(o,b):
    return masked_mean(F.cross_entropy(o['decision'].float(),b[6].long(),reduction='none'),b[19])

def action_loss(o,b):
    util=cost_aware_utilities(b[1],b[3],b[4]).detach()
    # Utilities supervise action quantiles; hard first-touch labels supervise decision probabilities.
    # Temperature calibration uses exactly these same hard labels.
    # Match the decision head target to temperature calibration and diagnostic reports.
    classification=masked_decision_loss(o,b)
    quantile=pinball(o['action_quantiles']/ret_scale,util/ret_scale)
    return CFG['decision_weight']*(classification+CFG['action_q_weight']*quantile)

def router_balance(o):
    usage=o['dispatch'].mean(0).detach()
    probs=o['router_probs'].mean(0)
    return len(usage)*(usage*probs).sum()

def forecast_losses(o,b,include_regime=True,include_paths=True,include_route=True):
    # Mandatory future outcomes in b are only read for LOSS, never forwarded into the model.
    r,d,m,n,v,decision,pattern,jump,path=b[1],b[2],b[3],b[4],b[5],b[6],b[8],b[9],b[10]
    barrier=b[16]
    total=(CFG['direction_weight']*F.cross_entropy(o['direction'].reshape(-1,3),d.reshape(-1))+
           CFG['quantile_weight']*pinball(o['quantiles']/ret_scale,r/ret_scale)+
           CFG['rv_weight']*F.smooth_l1_loss(torch.log1p(o['rv']/rv_scale),torch.log1p(v/rv_scale))+
           CFG['excursion_weight']*.5*(F.smooth_l1_loss(o['excursions'][...,0]/ret_scale,m/ret_scale)+
             F.smooth_l1_loss(o['excursions'][...,1]/ret_scale,n/ret_scale))+
           CFG['barrier_weight']*masked_barrier_loss(o,b)+
           CFG['pattern_weight']*F.binary_cross_entropy_with_logits(o['pattern'],pattern)+
           CFG['pattern_detail_weight']*F.binary_cross_entropy_with_logits(o['pattern_detail'],b[14])+
           CFG['jump_weight']*F.binary_cross_entropy_with_logits(o['jump'],jump)+
           CFG['path_weight']*F.smooth_l1_loss(o['path'],path))
    if include_paths:
        total=total+CFG['path_nll_weight']*path_distribution_nll(o,b[15])
    if include_regime:
        total=total+CFG['regime_weight']*regime_loss(o,b)
    if include_route:
        total=total+CFG['route_balance_weight']*router_balance(o)
    return total

def objective(o,b):
    return forecast_losses(o,b)+action_loss(o,b)

def joint_pcgrad_backward(o,b):
    # Two-objective PCGrad on shared parameters; private-head gradients remain intact.
    predictive=forecast_losses(o,b);decision_obj=action_loss(o,b)
    params=[p for p in model.parameters() if p.requires_grad]
    a=torch.autograd.grad(predictive,params,retain_graph=True,allow_unused=True)
    g=torch.autograd.grad(decision_obj,params,allow_unused=True)
    common=[i for i,(x,y) in enumerate(zip(a,g)) if x is not None and y is not None]
    if common:
        dot=sum((a[i]*g[i]).sum() for i in common)
        aa=sum(a[i].square().sum() for i in common).clamp(min=1e-12)
        gg=sum(g[i].square().sum() for i in common).clamp(min=1e-12)
        cosine=float((dot/torch.sqrt(aa*gg)).detach())
        conflict=bool(dot.detach().item()<0)
        alpha=(dot/gg).detach() if conflict else None
        beta=(dot/aa).detach() if conflict else None
    else:
        cosine=float('nan');conflict=False;alpha=beta=None
    for i,p in enumerate(params):
        x,y=a[i],g[i]
        if x is not None and y is not None and conflict:
            p.grad=(x-alpha*y+y-beta*x).detach()
        elif x is not None and y is not None:p.grad=(x+y).detach()
        elif x is not None:p.grad=x.detach()
        elif y is not None:p.grad=y.detach()
    return {'gradient_cosine':cosine,'gradient_conflict':conflict}

train_dl=dl(0,True,CFG['pool_assets']);val_dl=dl(1,False,CFG['pool_assets'])
outdir=Path(CFG['output_dir']);outdir.mkdir(parents=True,exist_ok=True)
STAGES=list(CFG['stage_epochs'])
assert STAGES==['A_SSL','B_REGIME','C_GRAPH','D_FORECAST','E_PATH','F_EXPERTS','G_DECISION','H_JOINT']
resume_path=outdir/'v7314_pooled_resume.pt';best_path=outdir/'v7314_pooled_H_JOINT_best.pt'
source_info=[(str(p),p.stat().st_size,p.stat().st_mtime_ns) for p in paths]
contract={'model':{'features':len(FEATURES),'hidden':CFG['hidden'],'experts':CFG['experts'],
            'topk':CFG['top_k'],'layers':CFG['transformer_layers'],'graph_layers':CFG['graph_layers'],
            'gru_layers':CFG['gru_layers'],'expert_width':CFG['expert_width'],
            'path_knots':CFG['path_knots'],'path_rank':CFG['path_rank'],
            'mtf_timeframes':CFG['mtf_timeframes'],'mtf_window':CFG['mtf_window'],
            'mtf_encoder_layers':CFG['mtf_encoder_layers'],
            'graph_history':(CFG['graph_history_points'],CFG['graph_history_stride']),
            'version':'v7314_TRAINING_QUALITY_GATES'},
          'horizons':H,'lookback':lb,'splits':[str(c) for c in cuts],
          'train_start':str(train_start),'train_end':str(train_end),
          'sampling':{k:CFG[k] for k in ('train_years','recent_half_life_days','recent_uniform_floor','stress_boost','stress_vol_quantile','stress_cap')},
          'assets':assets,'pooled_assets':CFG['pool_assets'],'target_asset':CFG['target_asset'],'context_assets':CFG['context_assets'],
          'cache_fingerprint':CACHE_FINGERPRINT,'context_scalers':context_scalers,'sources':source_info,'med':med.to_dict(),'iqr':iqr.to_dict(),
          'ret_scale':ret_scale,'rv_scale':rv_scale,
          'mtf_scalers':{k:{'med':v[0].to_dict(),'iqr':v[1].to_dict()} for k,v in mtf_scalers.items()},'stage_epochs':CFG['stage_epochs'],
          'stage_lrs':CFG['stage_lr_mult'],'stage_patience':CFG['stage_patience'],
          'stage_max_val_steps':CFG['stage_max_val_steps'],
          'max_train_steps_per_epoch':CFG['max_train_steps_per_epoch'],
          'loss_weights':{k:CFG[k] for k in CFG if k.endswith('_weight')},
          'seed':SEED,'optimizer':'AdamW','weight_decay':.01,
          'research_adverse_penalty':CFG['research_adverse_penalty'],
          'reference_cost_bps':CFG['reference_cost_bps'],
          'pcgrad':CFG['use_pcgrad_joint'],
          'precision':'bf16' if USE_BF16 else 'fp32','loader_workers':CFG['loader_workers'],
          'graph_index_cache':CFG['graph_index_cache'],
          'checkpoint_every_steps':CFG['checkpoint_every_steps'],
          'validation_report_months':CFG['validation_report_months'],
          'diagnostic_gradient_assert':CFG['diagnostic_gradient_assert']}
contract_hash=hashlib.sha256(json.dumps(contract,sort_keys=True).encode()).hexdigest()

def select_stage_parameters(stage):
    temporal=('input','conv','encoder','longrange','ssm','graph_fusion','mtf')
    graph=('graph_input','graph','mtf')
    forecast=('mtf','direction','quantiles','rv','excursions','barrier','pattern','pattern_detail','jump','path','forecast_fusion')
    if stage=='A_SSL':prefixes=temporal+('ssl_predict',)
    elif stage=='B_REGIME':prefixes=temporal+('regime',)
    elif stage=='C_GRAPH':prefixes=graph+('graph_fusion','forecast_fusion','direction','quantiles')
    elif stage=='D_FORECAST':prefixes=temporal+graph+('regime',)+forecast
    elif stage=='E_PATH':prefixes=temporal+graph+('forecast_fusion','path_distribution','path')
    elif stage=='F_EXPERTS':prefixes=temporal+graph+('regime','router','experts','path_distribution')+forecast
    elif stage=='G_DECISION':prefixes=('decision','decision_quantiles')
    elif stage=='H_JOINT':prefixes=tuple({s.split('.')[0] for s,_ in model.named_parameters()})
    else:raise ValueError(stage)
    for name,p in model.named_parameters():p.requires_grad=name.split('.')[0] in prefixes
    active=sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert active>0
    return active

# Fail-fast check for critical stage-to-head gradient connections (one batch per stage).
STAGE_GRADIENT_HEADS={
    'A_SSL':('ssl_predict',),
    'B_REGIME':('regime',),
    'C_GRAPH':('direction','quantiles'),
    'D_FORECAST':('direction','quantiles','rv','barrier','path'),
    'E_PATH':('path_distribution','path'),
    'F_EXPERTS':('router','direction','path_distribution'),
    'G_DECISION':('decision','decision_quantiles'),
    'H_JOINT':('direction','decision','path_distribution','path'),
}
def assert_stage_head_gradients(stage):
    if not CFG['diagnostic_gradient_assert']:return {}
    diagnostic={}
    for head in STAGE_GRADIENT_HEADS[stage]:
        ps=[p for name,p in model.named_parameters() if name.startswith(head+'.')]
        if not ps:raise RuntimeError(f'{stage}: missing expected module {head}')
        if not any(p.requires_grad for p in ps):raise RuntimeError(f'{stage}: {head} is frozen but its loss requires training')
        grads=[p.grad for p in ps if p.grad is not None]
        if not grads:raise RuntimeError(f'{stage}: missing gradient to {head}')
        if any(not torch.isfinite(g).all().item() for g in grads):raise FloatingPointError(f'{stage}: nonfinite gradient in {head}')
        magnitude=sum(float(g.detach().float().abs().sum()) for g in grads)
        if magnitude<=0:raise RuntimeError(f'{stage}: zero gradient in {head}')
        diagnostic[head]=magnitude
    return diagnostic

def stage_step(stage,b):
    x=b[0]
    if stage=='A_SSL':
        o=model(x[:,:-1],None,None,use_experts=False,use_mtf=False)
        return F.smooth_l1_loss(o['ssl_prediction'],x[:,-1].clamp(-12,12))
    has_experts=stage in ('F_EXPERTS','G_DECISION','H_JOINT')
    graph_on=stage not in ('B_REGIME',)
    o=model(x,b[11] if graph_on else None,b[12] if graph_on else None,use_experts=has_experts,mtf=b[17],mtf_mask=b[18])
    if stage=='B_REGIME':return regime_loss(o,b)
    if stage=='C_GRAPH':
        return F.cross_entropy(o['direction'].reshape(-1,3),b[2].reshape(-1)) + pinball(o['quantiles']/ret_scale,b[1]/ret_scale)
    if stage=='D_FORECAST':return forecast_losses(o,b,include_paths=False,include_route=False)
    if stage=='E_PATH':return path_distribution_nll(o,b[15])+F.smooth_l1_loss(o['path'],b[10])
    if stage=='F_EXPERTS':return forecast_losses(o,b,include_paths=True,include_route=True)
    if stage=='G_DECISION':return action_loss(o,b)
    if stage=='H_JOINT':return objective(o,b)
    raise ValueError(stage)

def atomic_save(obj,path):
    temp=path.with_suffix(path.suffix+'.tmp');torch.save(obj,temp);temp.replace(path)

def rng_snapshot():
    snapshot={'torch':torch.get_rng_state(),'numpy':np.random.get_state(),'python':random.getstate()}
    if torch.cuda.is_available():snapshot['cuda']=torch.cuda.get_rng_state_all()
    return snapshot

def rng_restore(snapshot):
    torch.set_rng_state(snapshot['torch']);np.random.set_state(snapshot['numpy']);random.setstate(snapshot['python'])
    if torch.cuda.is_available() and 'cuda' in snapshot:torch.cuda.set_rng_state_all(snapshot['cuda'])

@torch.no_grad()
def expert_diagnostics(loader,limit=3):
    model.eval();router_list=[];dispatch_list=[];ablation=[];graph_ablation=[]
    for i,pack in enumerate(loader):
        if i>=limit:break
        b=move_batch(pack)
        o=model(b[0],b[11],b[12],mtf=b[17],mtf_mask=b[18]);loss=F.cross_entropy(o['direction'].reshape(-1,3),b[2].reshape(-1)).item()
        router_list.append(o['router_probs'].mean(0).cpu().numpy());dispatch_list.append(o['dispatch'].mean(0).cpu().numpy())
        no_graph=model(b[0],b[11],b[12],mtf=b[17],mtf_mask=b[18],use_graph=False)
        graph_ablation.append(F.cross_entropy(no_graph['direction'].reshape(-1,3),b[2].reshape(-1)).item()-loss)
        if CFG['validate_expert_ablation']:
            deltas=[]
            for idx in range(CFG['experts']):
                oa=model(b[0],b[11],b[12],mtf=b[17],mtf_mask=b[18],ablated_expert=idx)
                deltas.append(F.cross_entropy(oa['direction'].reshape(-1,3),b[2].reshape(-1)).item()-loss)
            ablation.append(deltas)
    model.train()
    if not router_list:return {'available':False}
    return {'available':True,'batches':len(router_list),
      'expert_router_prob':np.mean(router_list,axis=0).round(5).tolist(),
      'expert_actual_dispatch':np.mean(dispatch_list,axis=0).round(5).tolist(),
      'direction_loss_increase_without_each_expert':np.mean(ablation,axis=0).round(6).tolist() if ablation else None,
      'direction_loss_increase_without_graph':float(np.mean(graph_ablation)),
      'note':'Positive ablation delta indicates contribution on these validation batches only.'}

# Real-data gate: no optimizer step, no weight update; fail early on invalid LazyFold.
def verify_real_training_batches():
    try:
        train_pack=next(iter(train_dl))
        validation_pack=next(iter(val_dl))
    except StopIteration as exc:
        raise RuntimeError('Real-data preflight: empty train or validation loader') from exc
    for name,pack in (('train',train_pack),('validation',validation_pack)):
        if len(pack)!=21 or pack[0].shape[1:]!=(lb,len(FEATURES)):
            raise RuntimeError(f'Real-data preflight: invalid {name} batch structure')
        if pack[17].shape[1:]!=(len(TF_NAMES),CFG['mtf_window'],len(FEATURES)):
            raise RuntimeError(f'Real-data preflight: invalid {name} MTF shape')
        for index in (0,1,11,15,17):
            if not torch.isfinite(pack[index]).all():
                raise FloatingPointError(f'Real-data preflight: nonfinite {name} tensor {index}')
    saved_rng=rng_snapshot()
    previous_requires={name:p.requires_grad for name,p in model.named_parameters()}
    was_training=model.training
    try:
        select_stage_parameters('H_JOINT')
        model.train()
        model.zero_grad(set_to_none=True)
        b=move_batch(train_pack)
        with amp_context():
            outputs=model(b[0],b[11],b[12],mtf=b[17],mtf_mask=b[18])
            loss=objective(outputs,b)
        if not torch.isfinite(loss):
            raise FloatingPointError('Real-data preflight: nonfinite H_JOINT loss')
        with amp_context():
            diagnostic=joint_pcgrad_backward(outputs,b)
        assert_stage_head_gradients('H_JOINT')
        for group in ('mtf','experts'):
            gradients=[p.grad for name,p in model.named_parameters()
                       if name.startswith(group+'.') and p.grad is not None]
            if any(not torch.isfinite(g).all().item() for g in gradients):
                raise FloatingPointError(f'Real-data preflight: nonfinite {group} gradient')
            if not gradients or not any(torch.count_nonzero(g).item()>0 for g in gradients):
                print('REAL-DATA PREFLIGHT WARNING: no gradient',group)
        model.eval()
        with torch.no_grad(),amp_context():
            validation_loss=stage_step('H_JOINT',move_batch(validation_pack))
        if not torch.isfinite(validation_loss):
            raise FloatingPointError('Real-data preflight: nonfinite validation loss')
        print('REAL-DATA PREFLIGHT PASS',{'train_batch':len(train_pack[0]),
            'validation_batch':len(validation_pack[0]),
            'train_loss':float(loss.detach()),'validation_loss':float(validation_loss),
            'gradient_cosine':diagnostic['gradient_cosine']})
    finally:
        model.zero_grad(set_to_none=True)
        for name,p in model.named_parameters():
            p.requires_grad=previous_requires[name]
        model.train(was_training)
        rng_restore(saved_rng)

verify_real_training_batches()

history=[];stage_index=0;epoch_in_stage=0;stage_best=float('inf');bad=0;checkpoint=None;batch_in_epoch=0;partial_sum=0.;partial_seen=0
if resume_path.exists():
    checkpoint=torch.load(resume_path,map_location='cpu',weights_only=False)
    if checkpoint['contract_hash']!=contract_hash:raise RuntimeError('Resume data/model/training contract mismatch')
    model.load_state_dict(checkpoint['model']);stage_index=checkpoint['stage_index']
    epoch_in_stage=checkpoint['epoch_in_stage'];stage_best=checkpoint['stage_best']
    bad=checkpoint['bad'];history=checkpoint['history'];rng_restore(checkpoint['rng'])
    batch_in_epoch=checkpoint.get('batch_in_epoch',0);partial_sum=checkpoint.get('partial_sum',0.);partial_seen=checkpoint.get('partial_seen',0)
    print('Resuming:',STAGES[stage_index] if stage_index<len(STAGES) else 'COMPLETE')
for si in range(stage_index,len(STAGES)):
    stage=STAGES[si];active=select_stage_parameters(stage)
    lr=CFG['lr']*CFG['stage_lr_mult'][stage]
    opt=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=lr,weight_decay=.01)
    if si==stage_index and (epoch_in_stage>0 or batch_in_epoch>0):
        if checkpoint.get('optimizer') is None:raise RuntimeError('Resume missing optimizer')
        opt.load_state_dict(checkpoint['optimizer'])
    else:epoch_in_stage=0;stage_best=float('inf');bad=0;batch_in_epoch=0;partial_sum=0.;partial_seen=0
    stage_best_path=outdir/f'v7314_pooled_{stage}_best.pt'
    print('Stage',stage,'trainable',active,'/',PARAMETERS,'lr',lr)
    for epoch in range(epoch_in_stage,int(CFG['stage_epochs'][stage])):
        model.train();train_sum=partial_sum;train_seen=partial_seen;cosines=[];conflicts=0
        offset=batch_in_epoch if (si==stage_index and epoch==epoch_in_stage) else 0
        train_dl.sampler.set_epoch('pooled',stage,epoch,offset)
        for step,pack in enumerate(train_dl,start=offset):
            b=move_batch(pack)
            opt.zero_grad(set_to_none=True)
            with amp_context():
                if stage=='H_JOINT' and CFG['use_pcgrad_joint']:
                    o=model(b[0],b[11],b[12],mtf=b[17],mtf_mask=b[18]);loss=objective(o,b)
                else:
                    loss=stage_step(stage,b)
            if not torch.isfinite(loss):raise FloatingPointError(f'{stage} nonfinite loss')
            if stage=='H_JOINT' and CFG['use_pcgrad_joint']:
                # Recalculate objective gradients in BF16 autocast; gradients stay FP32.
                with amp_context():diag=joint_pcgrad_backward(o,b)
                if np.isfinite(diag['gradient_cosine']):cosines.append(diag['gradient_cosine'])
                conflicts+=int(diag['gradient_conflict'])
            else:
                loss.backward()
            if not torch.isfinite(loss):raise FloatingPointError(f'{stage} nonfinite loss')
            if step==0:stage_gradient_probe=assert_stage_head_gradients(stage)
            norm=nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad),1.)
            if not torch.isfinite(norm):raise FloatingPointError(f'{stage} nonfinite gradient')
            opt.step();train_sum+=float(loss.detach())*len(b[0]);train_seen+=len(b[0])
            if CFG['checkpoint_every_steps'] and (step+1)%CFG['checkpoint_every_steps']==0:
                atomic_save({'model':model.state_dict(),'optimizer':opt.state_dict(),'contract_hash':contract_hash,
                    'stage_index':si,'epoch_in_stage':epoch,'batch_in_epoch':step+1,
                    'partial_sum':train_sum,'partial_seen':train_seen,
                    'stage_best':stage_best,'bad':bad,'history':history,'rng':rng_snapshot()},resume_path)
            if CFG['max_train_steps_per_epoch'] is not None and step+1>=CFG['max_train_steps_per_epoch']:break
        model.eval();val_sum=0.;val_seen=0
        with torch.no_grad():
            for step,pack in enumerate(val_dl):
                b=move_batch(pack)
                with amp_context():loss=stage_step(stage,b)
                if not torch.isfinite(loss):raise FloatingPointError(f'{stage} nonfinite validation')
                val_sum+=float(loss)*len(b[0]);val_seen+=len(b[0])
                if CFG['stage_max_val_steps'] is not None and step+1>=CFG['stage_max_val_steps']:break
        if train_seen==0 or val_seen==0:raise RuntimeError('Empty train/val fold')
        score=val_sum/val_seen
        entry={'stage':stage,'epoch':epoch+1,'train_loss':train_sum/train_seen,
               'validation_loss':score,'active_parameters':active,'train_seen':train_seen,
               'validation_seen':val_seen,'lr':lr,
               'gradient_cosine':float(np.mean(cosines)) if cosines else None,
               'gradient_conflict_steps':conflicts}
        if stage in ('C_GRAPH','F_EXPERTS','H_JOINT'):
            entry['expert_graph_diagnostics']=expert_diagnostics(val_dl,CFG['expert_val_batches'])
        history.append(entry);print(entry)
        if score<stage_best-1e-6:
            stage_best=score;bad=0
            atomic_save({'model':model.state_dict(),'contract_hash':contract_hash,
                         'stage':stage,'epoch':epoch+1,'validation_loss':score},stage_best_path)
        else:bad+=1
        atomic_save({'model':model.state_dict(),'optimizer':opt.state_dict(),
                     'contract_hash':contract_hash,'stage_index':si,'epoch_in_stage':epoch+1,
                     'stage_best':stage_best,'bad':bad,'history':history,'rng':rng_snapshot(),
                     'batch_in_epoch':0,'partial_sum':0.,'partial_seen':0},resume_path)
        batch_in_epoch=0;partial_sum=0.;partial_seen=0
        if bad>=CFG['stage_patience'][stage]:break
    if not stage_best_path.exists():raise RuntimeError(f'Missing checkpoint: {stage_best_path}')
    model.load_state_dict(torch.load(stage_best_path,map_location='cpu',weights_only=False)['model'])
    atomic_save({'model':model.state_dict(),'optimizer':None,'contract_hash':contract_hash,
                 'stage_index':si+1,'epoch_in_stage':0,'stage_best':float('inf'),
                 'bad':0,'history':history,'rng':rng_snapshot(),
                 'batch_in_epoch':0,'partial_sum':0.,'partial_seen':0},resume_path)
    epoch_in_stage=0;batch_in_epoch=0;partial_sum=0.;partial_seen=0
if not best_path.exists():raise RuntimeError('Final joint checkpoint not available')
model.load_state_dict(torch.load(best_path,map_location='cpu',weights_only=False)['model'])
model.eval();print('Pooled A-H complete; BTC-only validation follows; sealed OOS remains locked.')
@torch.no_grad()
def btc_selection_quality():
    # BTC selection report for the frozen pooled checkpoint; report/OOS are independent.
    was_training=model.training
    model.eval()
    seen=0; direction_sum=0.; decision_sum=0.; decision_n=0; path_sum=0.
    for pack in dl(1,False,[CFG['target_asset']],validation_role='selection'):
        b=move_batch(pack)
        with amp_context():o=model(b[0],b[11],b[12],mtf=b[17],mtf_mask=b[18])
        z=len(b[0]);seen+=z
        direction_sum+=float(F.cross_entropy(o['direction'].float().reshape(-1,3),b[2].reshape(-1),reduction='sum'))
        decision_loss=F.cross_entropy(o['decision'].float(),b[6].long(),reduction='none')
        valid=b[19].float();decision_sum+=float((decision_loss*valid).sum());decision_n+=int(valid.sum())
        path_sum+=float(path_distribution_nll(o,b[15]))*z
    model.train(was_training)
    if not seen or not decision_n:raise RuntimeError('BTC quality guard has no valid selection examples')
    return {'n':seen,'decision_n':decision_n,'direction_ce':direction_sum/(seen*len(H)),
            'decision_ce':decision_sum/decision_n,'path_nll':path_sum/seen}


pooled_btc_quality=btc_selection_quality()
print('BTC quality at pooled checkpoint:',pooled_btc_quality)

# SINGLE PHASE: pooled A-H checkpoint is final; BTC-only validation follows.
model.eval()
(outdir/'pooled_btc_selection_quality.json').write_text(json.dumps(pooled_btc_quality,indent=2))
val_dl=dl(1,False,[CFG['target_asset']])
cal_dl=dl(2,False,[CFG['target_asset']])
sealed_dl=dl(3,False,[CFG['target_asset']])
print('Pooled A-H complete; BTC-only validation/calibration/OOS; sealed OOS locked.')


## Calibration and sealed evaluation
- Stage H best model selected on Validation only.
- Separate Calibration partition uses streaming multiclass log-loss over a temperature grid; **no full-output arrays in RAM**.
- Optional sealed OOS reads one batch at a time; never used to tune models or temperatures.
- Sample signals limited to a few examples; no order execution or broker-specific risk calculations.


In [ ]:
# Streaming temperature calibration and validation diagnostics. Sealed data never accessed by default.
from scipy.special import softmax
from sklearn.metrics import balanced_accuracy_score
@torch.no_grad()
def streaming_temperature_fit():
    temps=np.geomspace(.4,4.,40)
    totals=np.zeros(len(temps),dtype=np.float64);seen=0
    model.eval()
    for pack in dl(2,False,[CFG['target_asset']]):
        b=move_batch(pack)
        logits=model(b[0],b[11],b[12],mtf=b[17],mtf_mask=b[18])['decision']
        target=b[6]
        mask=b[19].bool()
        if not mask.any():continue
        logits=logits[mask];target=target[mask]
        for j,t in enumerate(temps):
            totals[j]+=float(F.cross_entropy(logits/t,target,reduction='sum'))
        seen+=len(target)
    if not seen:raise RuntimeError('Calibration set has no examples')
    return float(temps[np.argmin(totals)]),seen,float(np.min(totals)/seen)
TEMPERATURE,N_CAL,CAL_LOGLOSS=streaming_temperature_fit()
(outdir/'calibration.json').write_text(json.dumps({'temperature':TEMPERATURE,'n_calibration':N_CAL,
    'multiclass_log_loss':CAL_LOGLOSS,'contract_hash':contract_hash,
    'frozen_checkpoint':str(best_path),'training_mode':'pooled_A_to_H_only'},indent=2))
print('Decision temperature:',TEMPERATURE,'calibration_n:',N_CAL)

@torch.no_grad()
def validation_quality_report(limit_batches=None):
    n=0;direction_correct=0;baseline_correct=0;mae=0.;baseline_mae=0.;q10cover=0;q90cover=0
    path_nll=0.;graph_loss_delta=0.;regime_abs_error=0.;regime_denominator=0
    model.eval()
    for j,pack in enumerate(dl(1,False,[CFG['target_asset']],validation_role='report')):
        if limit_batches is not None and j>=limit_batches:break
        b=move_batch(pack)
        o=model(b[0],b[11],b[12],mtf=b[17],mtf_mask=b[18]);no_graph=model(b[0],b[11],b[12],mtf=b[17],mtf_mask=b[18],use_graph=False)
        true=b[1][:,-1];pred=o['quantiles'][:,-1]
        direction_correct+=int((o['direction'][:,-1].argmax(-1)==b[2][:,-1]).sum())
        baseline_correct+=int((b[2][:,-1]==1).sum())
        mae+=float((pred[:,1]-true).abs().sum());baseline_mae+=float(true.abs().sum())
        q10cover+=int((true<=pred[:,0]).sum());q90cover+=int((true<=pred[:,2]).sum())
        path_nll+=float(path_distribution_nll(o,b[15]))*len(b[0])
        graph_loss_delta+=(float(F.cross_entropy(no_graph['direction'].reshape(-1,3),b[2].reshape(-1)))-
                           float(F.cross_entropy(o['direction'].reshape(-1,3),b[2].reshape(-1))))*len(b[0])
        regime_abs_error+=float(((torch.sigmoid(o['regime'])-b[7]).abs()*b[13]).sum())
        regime_denominator+=float(b[13].sum())
        n+=len(b[0])
    if not n:raise RuntimeError('No validation data')
    return {'validation_samples':n,'direction_accuracy':direction_correct/n,
        'always_wait_direction_accuracy':baseline_correct/n,
        'median_forecast_mae':mae/n,'naive_zero_return_mae':baseline_mae/n,
        'q10_empirical_coverage':q10cover/n,'q90_empirical_coverage':q90cover/n,
        'future_path_increment_nll':path_nll/n,
        'direction_log_loss_increase_without_graph':graph_loss_delta/n,
        'masked_regime_mean_absolute_error':regime_abs_error/max(1.,regime_denominator),
        'note':'Separated late-validation report cohort; earlier validation only used for checkpoint selection. NOT retrained walk-forward or sealed OOS.'}
validation_report=validation_quality_report()
(outdir/'validation_quality.json').write_text(json.dumps(validation_report,indent=2))
print('Validation diagnostics:',json.dumps(validation_report,indent=2))

@torch.no_grad()
def training_quality_diagnostics():
    # Independent late-validation report cohort: descriptive only, never used to select weights.
    model.eval();nh=len(H);n=0;dlog=np.zeros(nh);dcorr=np.zeros(nh);dmajor=np.zeros(nh)
    rmae=np.zeros(nh);zero_mae=np.zeros(nh);q10=np.zeros(nh);q90=np.zeros(nh)
    confusion=np.zeros((3,3),dtype=np.int64);d_brier=0.;d_ll=0.;d_count=0
    dispatch=[];routes=[]
    for pack in dl(1,False,[CFG['target_asset']],validation_role='report'):
        b=move_batch(pack)
        with amp_context():o=model(b[0],b[11],b[12],mtf=b[17],mtf_mask=b[18])
        B=len(b[0]);n+=B
        direction=o['direction'].float();real=b[2]
        dlog+=F.cross_entropy(direction.reshape(-1,3),real.reshape(-1),reduction='none').reshape(B,nh).sum(0).cpu().numpy()
        pred=direction.argmax(-1)
        dcorr+=(pred==real).sum(0).cpu().numpy()
        for j in range(nh):
            counts=torch.bincount(real[:,j],minlength=3)
            dmajor[j]+=int(counts.max())
        med=o['quantiles'].float()[...,1];ret=b[1].float()
        rmae+=(med-ret).abs().sum(0).cpu().numpy();zero_mae+=ret.abs().sum(0).cpu().numpy()
        q10+=(ret<=o['quantiles'][...,0]).sum(0).cpu().numpy()
        q90+=(ret<=o['quantiles'][...,2]).sum(0).cpu().numpy()
        valid=b[19].bool()
        if valid.any():
            logits=o['decision'].float()[valid]/TEMPERATURE
            label=b[6][valid].long();probs=F.softmax(logits,dim=-1)
            pred=probs.argmax(-1)
            for actual,predicted in zip(label.cpu().tolist(),pred.cpu().tolist()):confusion[actual,predicted]+=1
            d_ll+=float(F.cross_entropy(logits,label,reduction='sum'))
            d_brier+=float(((probs-F.one_hot(label,3).float())**2).sum())
            d_count+=len(label)
        dispatch.append((o['dispatch'].float().sum(0).cpu().numpy(),B))
        routes.append((o['router_probs'].float().sum(0).cpu().numpy(),B))
    if not n:raise RuntimeError('Empty independent validation report')
    c=confusion;support=c.sum(1);predicted=c.sum(0)
    precision=np.divide(c.diagonal(),predicted,out=np.zeros(3),where=predicted>0)
    recall=np.divide(c.diagonal(),support,out=np.zeros(3),where=support>0)
    f1=np.divide(2*precision*recall,precision+recall,out=np.zeros(3),where=(precision+recall)>0)
    prior=support/max(1,d_count)
    baseline_brier=float(1.-np.square(prior).sum()) if d_count else None
    return {'cohort':'independent late BTC validation report; never checkpoint selection',
        'samples':n,'horizons':H,
        'direction_ce':(dlog/n).tolist(),'direction_accuracy':(dcorr/n).tolist(),
        'batch_majority_accuracy_upper_bound':(dmajor/n).tolist(),
        'median_mae':(rmae/n).tolist(),'zero_return_mae':(zero_mae/n).tolist(),
        'q10_coverage':(q10/n).tolist(),'q90_coverage':(q90/n).tolist(),
        'decision_samples':d_count,'decision_confusion_rows_truth_cols_pred':c.tolist(),
        'decision_macro_f1':float(f1[support>0].mean()) if any(support>0) else None,
        'decision_log_loss':d_ll/max(1,d_count),'decision_brier':d_brier/max(1,d_count),
        'decision_class_prior_brier':baseline_brier,
        'expert_dispatch_frequency':(sum(v for v,_ in dispatch)/n).tolist(),
        'expert_router_probability':(sum(v for v,_ in routes)/n).tolist()}
training_quality_report=training_quality_diagnostics()
(outdir/'training_quality_diagnostics.json').write_text(json.dumps(training_quality_report,indent=2))
print('TRAINING QUALITY DIAGNOSTICS:',json.dumps(training_quality_report,indent=2))

@torch.no_grad()
def evaluate_sealed_streaming():
    if not CFG['run_sealed_test']:
        raise RuntimeError('Sealed OOS must be explicitly enabled after model/threshold freeze')
    model.eval();n=0;loss_sum=0.;brier_sum=0.;correct=np.zeros(3);true_count=np.zeros(3)
    pred_count=np.zeros(3);trade_count=0;signals_raw=[]
    ds=LazyFold(3,[CFG['target_asset']]);offset=0
    for pack in dl(3,False,[CFG['target_asset']]):
        b=move_batch(pack);o=model(b[0],b[11],b[12],mtf=b[17],mtf_mask=b[18])
        decision_ok=b[19].bool()
        if not decision_ok.any():
            offset+=len(b[0]);continue
        logits=o['decision'][decision_ok]/TEMPERATURE;prob=F.softmax(logits,dim=-1)
        pred=prob.argmax(-1);true=b[6][decision_ok]
        loss_sum+=float(F.cross_entropy(logits,true,reduction='sum'))
        brier_sum+=float(((prob-F.one_hot(true,3).float())**2).sum())
        signed=torch.where(pred==2,1.,torch.where(pred==0,-1.,0.))
        active=pred!=1
        trade_count+=int(active.sum())
        for k in range(3):
            true_count[k]+=int((true==k).sum());pred_count[k]+=int((pred==k).sum())
            correct[k]+=int(((pred==k)&(true==k)).sum())
        # Keep a bounded number of diagnostics for signal examples only.
        if len(signals_raw)<5:
            q=path_forecast_quantiles(o).detach().cpu().numpy()
            valid_positions=torch.nonzero(decision_ok,as_tuple=True)[0].cpu().tolist()
            for j in valid_positions[:max(0,5-len(signals_raw))]:
                signals_raw.append((ds.ref(offset+j),
                    {k:v[j].detach().cpu().numpy() for k,v in o.items()},q[j]))
        offset+=len(b[0]);n+=len(true)
    if not n:raise RuntimeError('Sealed OOS empty')
    recall=np.divide(correct,true_count,out=np.zeros_like(correct),where=true_count>0)
    return ({'samples':n,'decision_log_loss':loss_sum/n,'brier_score':brier_sum/n,
       'balanced_accuracy':float(recall[true_count>0].mean()),
       'true_class_counts':true_count.astype(int).tolist(),
       'pred_class_counts':pred_count.astype(int).tolist(),
       'overlapping_horizons_not_trade_backtest':True,
       'research_reference_cost_bps':CFG['reference_cost_bps'],
       'n_nonwait_signals':trade_count,'overlapping_returns_not_backtest':True,
       'sealed_evaluation_is_one_time':True},signals_raw)
metrics=None;sealed_examples=[]
if CFG['run_sealed_test']:
    metrics,sealed_examples=evaluate_sealed_streaming()
    print('SEALED OOS:',json.dumps(metrics,indent=2))
else:print('Sealed OOS locked. Calibration and validation reports saved.')


In [ ]:
# Broker-independent signal contract with correlated future path distribution and action quantiles.
CLASSES=['SHORT','WAIT','LONG']
REGIME_NAMES=['trend_persistence','high_volatility','transition','liquidity_cost',
              'past_jump','market_complexity','cross_asset_coupling']
PATTERN_TYPES=['breakout','false_breakout','reversal','trend_continuation']

def make_research_signal(ref,o,scenario_quantiles):
    (asset,asof),row=ref,o
    probs=softmax(row['decision']/TEMPERATURE,axis=-1)
    patterns=np.concatenate([row['pattern'][:,None],row['pattern_detail']],axis=-1)
    return {'schema':'manticore.quant_signal.v3.0.full',
      'asset':asset,'asof_utc':asof,'timeframes':['M5']+TF_NAMES,'decision':CLASSES[int(probs.argmax())],
      'decision_probabilities':dict(zip(CLASSES,np.round(probs,6).tolist())),
      'forecast_horizons_bars':H,'future_path_knots_bars':CFG['path_knots'],
      'forecast':{
         'direction_probabilities':softmax(row['direction'],axis=-1).round(6).tolist(),
         'log_return_quantiles_p10_p50_p90':row['quantiles'].round(7).tolist(),
         'future_realized_variance':row['rv'].round(10).tolist(),
         'first_touch_probs_by_horizon':softmax(row['barrier'],axis=-1).round(6).tolist(),
         'pattern_names':PATTERN_TYPES,
         'pattern_event_probabilities':(1/(1+np.exp(-patterns))).round(6).tolist(),
         'jump_event_probabilities':(1/(1+np.exp(-row['jump']))).round(6).tolist(),
         'correlated_path_log_return_quantiles':np.asarray(scenario_quantiles).round(7).tolist(),
         'path_distribution_family':'low_rank_multivariate_normal_log_increments'},
      'market_state':{'regime_names':REGIME_NAMES,
         'regime_probabilities':(1/(1+np.exp(-row['regime']))).round(6).tolist(),
         'expert_weights':row['gate'].round(5).tolist(),
         'mtf_attention_weights':dict(zip(TF_NAMES,row['mtf_attention'].round(5).tolist())),
         'actual_routed_experts':np.where(row['dispatch']>0)[0].tolist()},
      'decision_action_log_utility_quantiles':row['action_quantiles'].round(7).tolist(),
      'uncertainty':{'decision_entropy':float(-(probs*np.log(np.maximum(probs,1e-12))).sum())},
      'evidence':{'model':'ananta-v7.3.1.4-btc-only',
                  'calibration_set_separate':True,'reference_cost_assumption_bps':CFG['reference_cost_bps'],
                  'real_broker_fills':False,'execution_authorized':False},
      'execution_authorized':False}

signals=[make_research_signal(ref,row,q) for ref,row,q in sealed_examples]
assert all(s['execution_authorized'] is False for s in signals)
if signals:print(json.dumps(signals[0],indent=2))


In [ ]:
# Save final model. Path/decision heads included; no broker action.
outdir=Path(CFG['output_dir']);outdir.mkdir(parents=True,exist_ok=True)
atomic_save({'model_state':model.state_dict(),
    'model_config':{'features':len(FEATURES),'hidden':CFG['hidden'],'nexperts':CFG['experts'],
        'topk':CFG['top_k'],'nh':len(H),'layers':CFG['transformer_layers'],
        'heads':CFG['attention_heads'],'gru_layers':CFG['gru_layers'],
        'graph_layers':CFG['graph_layers'],'expert_width':CFG['expert_width'],
        'graph_history_points':CFG['graph_history_points'],
        'graph_history_stride':CFG['graph_history_stride'],
        'path_knots':CFG['path_knots'],'path_rank':CFG['path_rank']},
    'features':FEATURES,'horizons':H,'med':med.to_dict(),'iqr':iqr.to_dict(),
    'ret_scale':ret_scale,'rv_scale':rv_scale,'contract_hash':contract_hash,
    'context_scalers':context_scalers,
    'mtf_scalers':{k:{'med':v[0].to_dict(),'iqr':v[1].to_dict()} for k,v in mtf_scalers.items()},
    'calibration_temperature':TEMPERATURE,'research_only':True,'no_broker_execution':True,
    'training_mode':'pooled_A_to_H_only','stage_epochs':CFG['stage_epochs'],
     'pooled_assets':CFG['pool_assets'],'target_asset':CFG['target_asset']},
    outdir/'ananta_v7314_pooled_btc.pt')
if metrics is not None:(outdir/'sealed_oos_metrics.json').write_text(json.dumps(metrics,indent=2))
(outdir/'research_signals.jsonl').write_text('\n'.join(json.dumps(s) for s in signals)+'\n')
(outdir/'training_history.json').write_text(json.dumps({'pooled':history},indent=2))
(outdir/'data_coverage.json').write_text(json.dumps(coverage,indent=2))
print('Saved research artifacts:',outdir)


## Strict BTC-only contract

- Source, train, validation, calibration and sealed holdout: BTCUSD only.
- No auxiliary assets, NSX, contextual market data, or external ZIP prerequisites.
- Dynamic Graph architecture is retained but the external-neighbor path is not activated when its neighbor tensor has zero entries.
- Eight training stages and 28-epoch maximum retained without changing model dimensions.


## L4 and replay controls

- BF16 autocast where available, two DataLoader workers, pinned memory, memmap cross-asset indices.
- Single resumable pooled A-H training phase using `v7314_pooled_*` checkpoints; BTC-only validation and calibration follow training.
- Data windows preserved: 5y training + 6m validation + 3m calibration + 3m sealed holdout.
- Re-run all cells after changing data or source; checkpoint contract hashes enforce matching data splits and asset lists.


## Data quality and caveats

- Verify complete BTC historical coverage printed by the ingestion step.
- `aggTrades` are aggregate execution reports. They support aggressive-flow proxies but cannot prove liquidation cascades.
- NASDAQ closes on weekends/holidays and at session end; cross-market staleness is masked.
- External-market basis or premiums are not input features; BTCUSD is the sole market source.
- Dollar threshold bars are generated as events and converted to features on fixed M5; this is not yet an event-token Transformer.

### V7.3.1.4 reliability upgrade (2026-11-19)
- Strong SHA-256 checks for BTCUSD Binance ZIP files against Binance Vision CHECKSUM before reuse.
- Exact 5-minute continuity enforced for each feature segment, every lookback, all future labels and path knots; outages cannot become fixed-size price horizons.
- Independent masks for ambiguous Barrier and Decision labels; valid SSL/Forecast/Path samples are retained and not taught artificial WAIT.
- Train-only robust scaling combined with optional strictly causal rolling Z-scoring (200 observations, reset at time gaps).
- Deterministic per-epoch weighted sampling and step-level pooled/BTC checkpoint recovery (does not bypass the final unfinished partial batch).
- Split six-month validation into early checkpoint selection and later independent diagnostic period, separated by a temporal purge.
- Run on L4 with real archive data remains mandatory; passing static checks or synthetic tests is not a guarantee of convergent training.
- aggTrades cannot provide confirmed forced-liquidation labels or order book execution; Dollar Events still summarized to M5 rather than learned as separate tokens.
